<a href="https://colab.research.google.com/github/Karthikreddy1010/NYC-Taxi-Driver-ID-Misuse-Detection/blob/main/DIMD_Framework_Merged_Colab_LEAKFIXED_v5_perf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DIMD Framework — Driver Identity Misuse Detection
### Graph-Based Anomaly Detection for Driver Identity Misuse in Ride-Sharing Systems
**NYC Taxi Dataset 2023 · GraphSAGE · Heterogeneous GNN · Spatio-Temporal Signals**

This notebook merges the full DIMD pipeline (Phases I–VI) into one runnable Colab notebook, covering all **8 project objectives**:

| # | Objective | Phase |
|---|---|---|
| 1 | Detection Framework | I–V |
| 2 | Feature Engineering | II |
| 3 | Graph Construction | III |
| 4 | GNN vs Traditional Models | IV–V |
| 5 | Synthetic Anomaly Injection | II |
| 6 | Explainability | VI |
| 7 | Graph Structure Comparison | VI |
| 8 | Severity Scoring (GASS) | VI |

---
## ⚠️ Bug fixes applied in this merged version

This version fixes the issues found during code review against the methodology document:

1. **`torch.load` crash** — added `weights_only=False` for all `HeteroData` checkpoint loads (PyTorch ≥2.6 default changed).
2. **Train/test leakage (Obj 1, 4)** — driver-level temporal split rewritten so a driver belongs to **exactly one** of train/test, based on *first-trip* date, not "any trip in window" (which let almost every driver appear in both).
3. **Feature leakage** — driver-level aggregate features are now computed **separately** for the train window and the test window, instead of aggregating over the full year and then splitting drivers (the old code leaked future information into "past" features).
4. **GraphSAGE fallback split overlap** — fallback random split now resets `train_mask`/`test_mask` to all-`False` before assigning, so train/test never overlap.
5. **McNemar's test shape mismatch (Obj 4)** — all models now evaluated on the *same* aligned test driver index, so paired predictions line up.
6. **G-Device ablation crash (Obj 7)** — `G-Device` variant now also includes `driver↔driver` edges so the GraphSAGE encoder has at least one edge type with `driver` as destination (previously crashed: driver embeddings were never updated, `None` tensor in aggregation).
7. **No class-balanced training (Obj 4, 5)** — GraphSAGE now trains with a class-weighted loss (PC-GNN-style balancing), as specified in Methodology §6.1. This fixed a real failure mode: GraphSAGE was previously collapsing to predict "normal" for every account (0% recall on anomalies).
8. **GASS weight mismatch (Obj 8)** — severity weights corrected to match Methodology §7.3 exactly: GNN Prob **0.40** / Spatial Impossibility **0.30** / Temporal Persistence **0.20** / Device Switch **0.10** (was 0.40/0.25/0.20/0.15, and "Temporal Persistence" was never actually computed — it now is, from consecutive anomalous weeks).
9. **GASS/explanation cards used Z-scored features as raw units (Obj 6, 8)** — severity scoring and explanation cards now use a **separate, unscaled** copy of driver features, so "device switch rate" or "temporal shift" shown to investigators are real, interpretable numbers — not standardized scores that can be negative.
10. **Explanation cards mislabeled every flagged driver "High Risk" (Obj 6)** — card now reads its risk label from the actual GASS tier (Low/Medium/High) instead of a hardcoded string.
11. **LSTM split wasn't actually temporal (Obj 4)** despite the name — now uses the same first-trip-date driver split as the other baselines.
12. **Target/label leakage in baseline features (Obj 4)** — `temporal_persistence_weeks` is computed directly from the ground-truth `is_anomaly` label (longest run of consecutive anomalous weeks). It was included in `FEATURE_COLS` for LogReg/XGBoost/LSTM but correctly excluded from GraphSAGE's inputs — so the baselines were trained on a near-perfect proxy for the answer while the GNN wasn't, invalidating the Obj 4 comparison. Now excluded from every model's training features (still used legitimately, post-hoc, in GASS severity scoring, Obj 8).
13. **Partial/synthetic data made more visible (Obj 1)** — if no real TLC CSV is uploaded, the notebook already fell back to a small synthetic dataset, but this was easy to miss. A loud warning now prints whenever synthetic data is used, a `USING_REAL_DATA` flag is tracked, and it's echoed in the final run summary so results are never mistaken for the real 43M-row dataset. Default synthetic demo size increased (60k → 200k rows, 2k → 4k drivers) for a more representative demo run.

---
## ⚠️ Is this using the real 43M-row dataset, or synthetic data?

**By default, synthetic data**, unless you upload the real file. Look for the
`USING REAL DATA` / `USING SYNTHETIC DEMO DATA` line printed in Section 2 and
again in the final Section 18 summary — every number in this notebook is only
as real as that line says.

## How to use this notebook
1. Upload `2023_Yellow_Taxi_Trip_Data_20260401.csv` (NYC TLC Yellow Taxi 2023) when prompted in **Section 1**, *or* let the notebook auto-generate a small synthetic dataset with the same schema so you can test the full pipeline without the real file (see warning above — this is a demo/test mode, not a substitute for reporting real results).
2. Run all cells top to bottom (**Runtime → Run all**).
3. Outputs (figures, CSVs, model checkpoints, explanation cards) are written under `/content/dimd_project/`.


In [1]:
!pip install -q torch torch_geometric xgboost tqdm scipy scikit-learn matplotlib seaborn pandas numpy

In [2]:
# ============================================================
# SECTION 0 — Setup & Installation
# ============================================================
!pip install -q torch torch_geometric xgboost tqdm scipy scikit-learn matplotlib seaborn pandas numpy

import os
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import entropy, chi2
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve, f1_score,
    average_precision_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, Linear
from xgboost import XGBClassifier
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", palette="muted")

print("✅ Environment ready.")
print(f"   Torch: {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")


✅ Environment ready.
   Torch: 2.11.0+cu128  |  CUDA available: True


In [3]:
# ============================================================
# SECTION 1 — Central Configuration
# (mirrors original config.py; all pipeline cells import these names)
# ============================================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

PROJECT_ROOT = Path("/content/dimd_project")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ENRICHED_DIR = DATA_DIR / "enriched"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

for d in [RAW_DIR, PROCESSED_DIR, ENRICHED_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_CSV = PROJECT_ROOT / "/content/2023_Yellow_Taxi_Trip_Data_20260704.csv"
CLEAN_CSV = PROCESSED_DIR / "nyc_taxi_2023_clean.csv"
DRIVER_CSV = ENRICHED_DIR / "nyc_taxi_with_driver_id.csv"
DEVICE_CSV = ENRICHED_DIR / "nyc_taxi_with_devices.csv"
DEVICE_MAP_JSON = ENRICHED_DIR / "driver_device_mapping.json"
ANOMALY_CSV = ENRICHED_DIR / "nyc_taxi_with_anomalies.csv"
RIDE_FEATURES_CSV = PROCESSED_DIR / "nyc_taxi_with_features.csv"
DRIVER_FEATURES_CSV = PROCESSED_DIR / "driver_features.csv"          # Z-scored (for model training)
DRIVER_FEATURES_RAW_CSV = PROCESSED_DIR / "driver_features_raw.csv"  # FIX #9: unscaled (for GASS / explanations)
DRIVER_PROFILES_CSV = PROCESSED_DIR / "driver_profiles.csv"
GRAPH_PT = PROCESSED_DIR / "heterogeneous_graph.pt"
GRAPH_MAPPINGS_PKL = PROCESSED_DIR / "graph_mappings.pkl"
SCALER_PKL = MODELS_DIR / "feature_scaler.pkl"
SPLIT_JSON = PROCESSED_DIR / "driver_split.json"   # FIX #2/#5: single source of truth for train/test driver IDs

LSTM_MODEL = MODELS_DIR / "lstm_baseline.pth"
XGBOOST_MODEL = MODELS_DIR / "xgboost_baseline.pkl"
LOGREG_MODEL = MODELS_DIR / "logreg_baseline.pkl"
GRAPHSAGE_MODEL = MODELS_DIR / "graphsage_fraud_detector.pth"

# Use None for the full file; an int subsamples raw rows for faster iteration
SAMPLE_SIZE = None

# 🔧 FIX #3 (no silent fallback): if the temporal train/test split is
# infeasible (not enough calendar coverage), build_driver_split() will
# raise instead of silently swapping to a random split — UNLESS this is
# explicitly set True, in which case every downstream result is tagged
# "NON-TEMPORAL FALLBACK" via NON_TEMPORAL_FALLBACK_ACTIVE.
ALLOW_NON_TEMPORAL_FALLBACK = False
NON_TEMPORAL_FALLBACK_ACTIVE = False  # set by build_driver_split()


def print_fallback_banner_if_needed():
    """Call at the top of any results/report cell (Sections 13, 18, ...)."""
    if NON_TEMPORAL_FALLBACK_ACTIVE:
        print("\n" + "🚫" * 35)
        print("🚫 NON-TEMPORAL FALLBACK — results below are NOT a valid test of")
        print("🚫 generalization across time. Train/test drivers were split randomly,")
        print("🚫 not by first-trip date, because the loaded data didn't span enough")
        print("🚫 calendar time (see Section 7 warning). Do not report these numbers")
        print("🚫 as a generalization result — re-run with full calendar coverage.")
        print("🚫" * 35 + "\n")

# 🔧 FIX #13 (PARTIAL / DEMO DATA — flagged per user request): if no real TLC
# CSV is uploaded to RAW_CSV, Section 2 (`generate_synthetic_raw_data`) falls
# back to a SYNTHETIC dataset for the notebook to run end-to-end without the
# real file. That fallback is NOT the ~43M-record NYC TLC 2023 dataset the
# Methodology document describes — it is a small placeholder for pipeline
# testing/demo purposes only. `USING_REAL_DATA` (set in Section 2 once the
# loader runs) tracks which case you're in, and every results/report cell
# prints it so figures/metrics are never mistaken for real-data results.
# Increase SYNTHETIC_N_ROWS / TARGET_NUM_DRIVERS below for a larger demo run,
# but for a publishable result you must upload the real CSV to RAW_CSV.
SYNTHETIC_N_ROWS = 200_000   # was hard-coded to 60_000 inside the generator

# ── Synthetic Driver Configuration ──────────────────────────
TARGET_NUM_DRIVERS = 4_000
MIN_TRIPS_PER_DRIVER = 30
MAX_TRIPS_PER_DRIVER = 800
SHARED_ACCOUNT_RATE = 0.15

# ── Anomaly Injection (Objective 5) ─────────────────────────
IT_INJECTION_RATE = 0.10
BPS_INJECTION_RATE = 0.10

# 🔧 FIX #2 (injection redesign): severity mix per injected driver, plus a
# pool of "hard negative" NORMAL drivers who get 1-2 legitimately rare long
# trips (but stay labeled is_anomaly=0). Both changes exist so no single
# thresholded feature (e.g. "jump > X km") can separate classes near-perfectly.
ANOMALY_SEVERITY_MIX = {"mild": 0.34, "moderate": 0.33, "severe": 0.33}
HARD_NEGATIVE_RATE = 0.08          # fraction of NORMAL drivers given 1-2 rare-but-legit long trips
HARD_NEGATIVE_JUMP_MEAN_KM = 10.0  # legit rare trips overlap the "mild" injected-anomaly distance band
HARD_NEGATIVE_JUMP_STD_KM = 4.0

# ── Device Configuration ────────────────────────────────────
DEVICE_COUNT_PROBS = [0.75, 0.15, 0.08, 0.02]
DEVICE_SWITCH_PROB = 0.05

# ── Behavioural Profiling (Objective 2) ─────────────────────
BASELINE_WINDOW_DAYS = 60
DBSCAN_EPS = 0.01
DBSCAN_MIN_SAMPLES = 5

# ── Graph Construction (Objective 3) ────────────────────────
SPATIAL_OVERLAP_THRESHOLD = 3
TEMPORAL_OVERLAP_THRESHOLD = 0.5
TEMPORAL_RIDE_GAP_MINUTES = 120

# ── Model Hyperparameters ───────────────────────────────────
LSTM_SEQ_LEN = 7
LSTM_HIDDEN_SIZE = 64
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT = 0.3
LSTM_EPOCHS = 30
LSTM_LR = 1e-3
LSTM_BATCH_SIZE = 256

XGB_N_ESTIMATORS = 300
XGB_MAX_DEPTH = 6
XGB_LR = 0.05
XGB_SUBSAMPLE = 0.8

GNN_HIDDEN_CHANNELS = 64
GNN_NUM_LAYERS = 3
GNN_DROPOUT = 0.3
GNN_EPOCHS = 100
GNN_LR = 1e-3
GNN_WEIGHT_DECAY = 1e-4

# ── Temporal Train/Test Split (Methodology §7.1) ────────────
# FIX #2/#3: drivers are split by FIRST TRIP DATE so each driver belongs to
# exactly one of train/test, and driver-level features are computed
# separately per window so no future information leaks into "past" features.
TRAIN_CUTOFF_MONTH = 9     # Jan–Sep 2023 -> train
TEST_START_MONTH = 10      # Oct–Dec 2023 -> test

# ── GASS Severity Scoring (Objective 8) ─────────────────────
# FIX #8: weights now match Methodology §7.3 exactly (sum to 1.0)
GASS_WEIGHT_GNN_PROB = 0.40
GASS_WEIGHT_SPATIAL_IMPOSSIBLE = 0.30
GASS_WEIGHT_TEMPORAL_PERSISTENCE = 0.20
GASS_WEIGHT_DEVICE_SWITCH = 0.10
GASS_THRESHOLD_LOW = 0.40
GASS_THRESHOLD_HIGH = 0.70

# ── Evaluation ───────────────────────────────────────────────
PRECISION_AT_K = 100
RECALL_TARGET = 0.90

# ── Spatial Impossibility ────────────────────────────────────
IMPOSSIBLE_SPEED_MPH = 120

# ── NYC Borough mapping ──────────────────────────────────────
BOROUGH_RANGES = {
    "Manhattan": list(range(4, 13)) + list(range(13, 25)) + list(range(41, 44))
                 + list(range(45, 49)) + list(range(50, 53)) + [12, 13, 24, 25]
                 + list(range(74, 76)) + list(range(79, 91)) + list(range(100, 115))
                 + list(range(113, 165)) + list(range(186, 195)) + list(range(202, 235))
                 + list(range(236, 264)),
    "Brooklyn": list(range(11, 12)) + list(range(14, 18)) + list(range(21, 23))
                + list(range(25, 27)) + list(range(29, 36)) + list(range(39, 41))
                + list(range(47, 50)) + list(range(52, 58)) + list(range(60, 70))
                + list(range(71, 74)) + list(range(76, 79)) + list(range(89, 98))
                + list(range(97, 100)) + list(range(106, 113)) + list(range(123, 135))
                + list(range(149, 155)) + list(range(155, 160)) + list(range(177, 186))
                + list(range(188, 195)) + list(range(210, 218)) + list(range(225, 232))
                + list(range(240, 253)) + list(range(255, 260)),
    "Queens": list(range(2, 11)) + list(range(15, 21)) + list(range(27, 35))
              + list(range(38, 42)) + list(range(53, 58)) + list(range(64, 71))
              + list(range(73, 78)) + list(range(86, 92)) + list(range(95, 102))
              + list(range(117, 125)) + list(range(130, 140)) + list(range(145, 155))
              + list(range(157, 162)) + list(range(171, 180)) + list(range(196, 210))
              + list(range(215, 225)) + list(range(238, 248)) + list(range(252, 260)),
    "Bronx": list(range(3, 10)) + list(range(13, 20)) + list(range(31, 38))
             + list(range(46, 53)) + list(range(59, 70)) + list(range(78, 82))
             + list(range(94, 100)) + list(range(119, 128)) + list(range(136, 145))
             + list(range(159, 170)) + list(range(174, 185)) + list(range(199, 210))
             + list(range(212, 220)) + list(range(233, 240)) + list(range(247, 255)),
    "Staten Island": list(range(5, 8)) + list(range(22, 26)) + list(range(43, 46))
                     + list(range(55, 60)) + list(range(110, 115)) + list(range(156, 160))
                     + list(range(176, 180)) + list(range(187, 192)) + list(range(204, 208))
                     + list(range(245, 252)),
}


# 🔧 FIX v4 (PERFORMANCE): `get_borough` used to loop over all 5 boroughs
# and do a Python `in` scan of each borough's zone-ID list (~50-260 items)
# for every call. It was invoked via `.apply(...)` on EVERY row of the raw
# dataset (Section 3, twice per row: pickup + dropoff) — on the full
# uploaded/downloaded year of data that's tens of millions of row-level
# Python function calls, each doing multiple linear list scans. Precompute
# a flat zone_id -> borough dict ONCE here (O(num_zones), ~263 entries)
# so each lookup is an O(1) dict get, and Section 3 can use a single vectorized
# `.map()` instead of `.apply(lambda ...)`.
_ZONE_TO_BOROUGH = {zid: borough for borough, ids in BOROUGH_RANGES.items() for zid in ids}


def get_borough(location_id: int) -> str:
    return _ZONE_TO_BOROUGH.get(location_id, "Unknown")


print(f"✅ Config loaded. Project root: {PROJECT_ROOT}")


✅ Config loaded. Project root: /content/dimd_project


In [4]:
# ============================================================
# SECTION 2 — Phase I: Data Acquisition (Objective 1)
# ============================================================
# Upload the real NYC TLC 2023 Yellow Taxi CSV to RAW_CSV, or this cell
# auto-generates a small synthetic dataset with the same schema so the
# whole notebook is runnable end-to-end for testing/demo purposes.
#
# 🔧 FIX #15 (root cause of the Section 7 "TEMPORAL SPLIT INFEASIBLE" stop):
# the file previously placed at RAW_CSV ("2023_Yellow_Taxi_Trip_Data_....csv")
# is an export from the NYC Open Data portal (data.cityofnewyork.us), whose
# UI export caps the number of rows it returns. Because the export is
# ordered chronologically, that cap silently truncated the file to only
# Jan 1-23, 2023 — 23 days out of 365 — even though it's still called
# "2023 Yellow Taxi Trip Data". Every driver's first trip therefore fell
# in month 1, so build_driver_split() (Section 7) always finds 0 test
# drivers after the month-9 cutoff and correctly refuses to proceed
# (that refusal is a deliberate guard from FIX #3 — it is NOT a bug and
# should not be bypassed by flipping ALLOW_NON_TEMPORAL_FALLBACK).
#
# The actual fix is to load genuine full-year data. TLC's official
# CloudFront distribution (linked from
# https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) publishes
# one Parquet file per month with a stable, documented URL pattern:
#   https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{MM}.parquet
# This section now downloads all 12 months of 2023 directly from that
# source (caching each month locally so re-runs don't re-download),
# concatenates them, and — if SAMPLE_SIZE is set — subsamples EACH MONTH
# individually before concatenating rather than subsampling the combined
# frame afterward, so a small SAMPLE_SIZE still preserves full Jan-Dec
# calendar coverage instead of accidentally re-creating the same
# single-month problem this is meant to fix.
#
# This full-year download is tried FIRST. If it fails (no internet access,
# a blocked domain, etc.), the cell falls back to the original behavior:
# use the uploaded RAW_CSV if present, else generate the synthetic demo
# dataset. USING_REAL_DATA / DATA_SOURCE track which path was actually used
# so every downstream results cell can report it accurately.

TLC_BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
TLC_YEAR = 2023
TLC_MONTHLY_DIR = RAW_DIR / "yellow_tripdata_2023_monthly"
TLC_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)
FULL_YEAR_CACHE = RAW_DIR / "yellow_tripdata_2023_full_year.parquet"


def generate_synthetic_raw_data(n_rows: int = SYNTHETIC_N_ROWS, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Synthetic fallback matching the NYC TLC Yellow Taxi schema."""
    rng = np.random.default_rng(seed)
    start, end = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-12-31 23:59:59")
    pickup = start + pd.to_timedelta(rng.uniform(0, (end - start).total_seconds(), n_rows), unit="s")
    duration_min = np.clip(rng.exponential(15, n_rows), 1, 180)
    dropoff = pickup + pd.to_timedelta(duration_min, unit="m")
    trip_distance = np.clip(rng.exponential(3, n_rows), 0.1, 40)
    fare_amount = np.clip(trip_distance * 2.5 + rng.normal(5, 2, n_rows), 2.5, 400)
    tip_amount = np.clip(fare_amount * rng.uniform(0, 0.25, n_rows), 0, None)
    total_amount = fare_amount + tip_amount + 0.5
    passenger_count = rng.choice([1, 2, 3, 4, 5, 6], size=n_rows, p=[0.6, 0.2, 0.08, 0.06, 0.03, 0.03])
    return pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": dropoff,
        "trip_distance": trip_distance,
        "fare_amount": fare_amount,
        "tip_amount": tip_amount,
        "total_amount": total_amount,
        "passenger_count": passenger_count,
        "PULocationID": rng.integers(1, 264, n_rows),
        "DOLocationID": rng.integers(1, 264, n_rows),
        "VendorID": rng.choice([1, 2], n_rows),
        "RatecodeID": 1,
    })


def _download_file(url: str, dest: Path, timeout: int = 120) -> bool:
    import urllib.request
    import urllib.error
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = resp.read()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            f.write(data)
        tmp.rename(dest)
        return True
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError) as e:
        print(f"  ⚠ download failed for {url}: {e}")
        return False


def download_full_year_2023_yellow_taxi(months=range(1, 13),
                                         sample_size: int | None = SAMPLE_SIZE) -> pd.DataFrame | None:
    """
    🔧 FIX #15: pulls all 12 months of the official TLC 2023 Yellow Taxi
    Parquet files (caching each month locally), subsamples PER MONTH
    (not after concatenation) when `sample_size` is set so calendar
    coverage survives subsampling, and concatenates into one full-year
    DataFrame. Returns None (rather than raising) if the download/cache
    can't be completed, so the caller can fall back gracefully.
    """
    if FULL_YEAR_CACHE.exists():
        print(f"  Using cached full-year file → {FULL_YEAR_CACHE}")
        return pd.read_parquet(FULL_YEAR_CACHE)

    months = list(months)
    per_month_quota = max(1, sample_size // len(months)) if sample_size else None
    frames = []

    for m in months:
        month_str = f"{TLC_YEAR}-{m:02d}"
        local_path = TLC_MONTHLY_DIR / f"yellow_tripdata_{month_str}.parquet"
        url = f"{TLC_BASE_URL}/yellow_tripdata_{month_str}.parquet"

        if not local_path.exists():
            print(f"  Downloading {month_str} …")
            if not _download_file(url, local_path):
                print(f"  ⚠ Skipping {month_str} (download failed) — full-year coverage will have a gap")
                continue
        else:
            print(f"  Using cached {month_str}")

        try:
            df_month = pd.read_parquet(local_path)
        except Exception as e:
            print(f"  ⚠ Could not read {local_path.name}: {e}")
            continue

        if per_month_quota and len(df_month) > per_month_quota:
            df_month = df_month.sample(n=per_month_quota, random_state=RANDOM_SEED)
        print(f"    {month_str}: {len(df_month):,} rows kept")
        frames.append(df_month)

    if not frames:
        print("  ⚠ No monthly files could be downloaded or read — falling back.")
        return None

    full = pd.concat(frames, ignore_index=True)
    try:
        full.to_parquet(FULL_YEAR_CACHE, index=False)
        print(f"  ✅ Cached combined full-year file → {FULL_YEAR_CACHE}")
    except Exception as e:
        print(f"  ⚠ Could not write full-year cache ({e}); continuing without caching it.")
    print(f"  ✅ Combined {len(frames)}/{len(months)} months → {len(full):,} rows")
    return full


class NYCTaxiDataLoader:
    def __init__(self, csv_path: Path = RAW_CSV):
        self.csv_path = Path(csv_path)

    def load(self, sample_size: int | None = SAMPLE_SIZE) -> pd.DataFrame:
        global USING_REAL_DATA, DATA_SOURCE

        # 🔧 FIX #15: try genuine full-year TLC data FIRST.
        print("Attempting full-year 2023 TLC download (Section 2, FIX #15) …")
        df = download_full_year_2023_yellow_taxi(sample_size=sample_size)
        if df is not None:
            USING_REAL_DATA = True
            DATA_SOURCE = "tlc_full_year_download"
        elif self.csv_path.exists():
            USING_REAL_DATA = True
            DATA_SOURCE = "uploaded_csv"
            print(f"Falling back to uploaded file: Loading {self.csv_path.name} …")
            print("  ⚠ NOTE: if this CSV came from the NYC Open Data portal UI export rather than")
            print("  ⚠ TLC's own monthly Parquet files, it may be truncated to a partial date range")
            print("  ⚠ (this is exactly what caused the Section 7 temporal-split failure before).")
            df = pd.read_csv(self.csv_path, low_memory=False)
        else:
            USING_REAL_DATA = False
            DATA_SOURCE = "synthetic"
            print("⚠" * 35)
            print(f"⚠ No full-year download and '{self.csv_path.name}' NOT FOUND.")
            print(f"⚠ Generating a SYNTHETIC PLACEHOLDER dataset instead: {SYNTHETIC_N_ROWS:,} rows.")
            print(f"⚠ This is NOT the real NYC TLC 2023 dataset — it is a small demo dataset for")
            print(f"⚠ exercising the pipeline. Every AUC/F1/GASS number produced by this run describes")
            print(f"⚠ the SYNTHETIC data only and must not be reported as a real-data result.")
            print("⚠" * 35)
            df = generate_synthetic_raw_data()
            df.to_csv(self.csv_path, index=False)

        print(f"  Raw rows loaded: {len(df):,}")
        for col in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")
        for col in ["trip_distance", "total_amount", "fare_amount", "passenger_count",
                    "PULocationID", "DOLocationID", "VendorID"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # 🔧 FIX #15: sampling here now only fires as a safety net for the
        # uploaded-CSV / synthetic paths — the TLC download path already
        # sampled per-month above, so full-year coverage isn't lost here.
        if sample_size and len(df) > sample_size and DATA_SOURCE != "tlc_full_year_download":
            print(f"  Sampling {sample_size:,} rows (seed={RANDOM_SEED}) …")
            df = df.sample(n=sample_size, random_state=RANDOM_SEED).reset_index(drop=True)

        print(f"  Final shape: {df.shape}")
        return df

    def inspect(self, df: pd.DataFrame) -> None:
        print(f"\n{'='*70}\nDATA INSPECTION REPORT\n{'='*70}")
        print(f"Shape          : {df.shape}")
        print(f"Data source    : {DATA_SOURCE}")
        if "tpep_pickup_datetime" in df.columns:
            dt = df["tpep_pickup_datetime"]
            print(f"Date range     : {dt.min()} → {dt.max()}")
            months_present = sorted(dt.dt.month.unique().tolist())
            print(f"Months present : {months_present}")
            if len(months_present) < 12:
                print(f"  ⚠ Only {len(months_present)}/12 months present — the temporal split in "
                      f"Section 7 needs month > {TRAIN_CUTOFF_MONTH} data to form a test set.")
        print(f"Vendors        : {df['VendorID'].unique().tolist()}")
        print(f"Pickup zones   : {df['PULocationID'].nunique()} unique")
        missing = df.isnull().sum()
        missing = missing[missing > 0]
        print("Missing values :", "none" if missing.empty else f"\n{missing.to_string()}")


def check_data_quality(df: pd.DataFrame) -> bool:
    issues = []
    if (df["trip_distance"] < 0).any():
        issues.append(f"  ⚠ {(df['trip_distance'] < 0).sum():,} rides with negative distance")
    if "tpep_dropoff_datetime" in df.columns:
        bad = (df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]).sum()
        if bad:
            issues.append(f"  ⚠ {bad:,} rides where dropoff < pickup")
    if (df["fare_amount"] <= 0).any():
        issues.append(f"  ⚠ {(df['fare_amount'] <= 0).sum():,} rides with invalid fare")
    if issues:
        print("Data quality issues found:")
        for m in issues:
            print(m)
        return False
    print("✅ Data quality checks passed.")
    return True


USING_REAL_DATA = False   # set by NYCTaxiDataLoader.load()
DATA_SOURCE = "unknown"   # 🔧 NEW (FIX #15): "tlc_full_year_download" | "uploaded_csv" | "synthetic"
loader = NYCTaxiDataLoader()
rides_raw = loader.load()
loader.inspect(rides_raw)
check_data_quality(rides_raw)
print(f"\n✅ Data loaded — {len(rides_raw):,} rows ready for cleaning")
print(f"{'✅ USING REAL NYC TLC DATA' if USING_REAL_DATA else '⚠️  USING SYNTHETIC DEMO DATA — see warning above'}"
      f" (source: {DATA_SOURCE})")


Attempting full-year 2023 TLC download (Section 2, FIX #15) …
  Using cached full-year file → /content/dimd_project/data/raw/yellow_tripdata_2023_full_year.parquet
  Raw rows loaded: 38,310,226
  Final shape: (38310226, 20)

DATA INSPECTION REPORT
Shape          : (38310226, 20)
Data source    : tlc_full_year_download
Date range     : 2001-01-01 00:06:49 → 2024-01-03 19:42:57
Months present : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Vendors        : [2, 1, 6]
Pickup zones   : 263 unique
Missing values : 
passenger_count          1309356
RatecodeID               1309356
store_and_fwd_flag       1309356
congestion_surcharge     1309356
airport_fee             35315203
Airport_fee              4304379
Data quality issues found:
  ⚠ 2,475 rides where dropoff < pickup
  ⚠ 394,503 rides with invalid fare

✅ Data loaded — 38,310,226 rows ready for cleaning
✅ USING REAL NYC TLC DATA (source: tlc_full_year_download)


In [5]:
# ============================================================
# SECTION 3 — Phase I: Pre-processing (Objective 1, Methodology §3.2)
# ============================================================
# 🔧 FIX v4 (PERFORMANCE): pu_borough/do_borough now use the vectorized
# `_ZONE_TO_BOROUGH` map built in Section 1, instead of a per-row
# `.apply(get_borough)` — see the FIX v4 note next to `get_borough`.

AIRPORT_ZONES = {1, 132, 138}  # JFK, LaGuardia, Newark


def clean_nyc_taxi_data(rides: pd.DataFrame) -> pd.DataFrame:
    rides = rides.copy()
    initial = len(rides)
    print(f"Initial records : {initial:,}")

    for col in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
        if col in rides.columns:
            rides[col] = pd.to_datetime(rides[col], errors="coerce")
    for col in ["trip_distance", "total_amount", "fare_amount", "passenger_count",
                "PULocationID", "DOLocationID", "VendorID"]:
        if col in rides.columns:
            rides[col] = pd.to_numeric(rides[col], errors="coerce")

    rides = rides.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                                  "PULocationID", "DOLocationID"])
    print(f"  After null datetime/location removal: {len(rides):,}")

    rides = rides[(rides["fare_amount"] > 0) & (rides["fare_amount"] <= 500)]
    print(f"  After fare filter (0 < fare ≤ $500): {len(rides):,}")

    is_airport = rides["PULocationID"].isin(AIRPORT_ZONES) | rides["DOLocationID"].isin(AIRPORT_ZONES)
    rides = rides[~((rides["trip_distance"] == 0) & ~is_airport)]
    print(f"  After zero-distance non-airport removal: {len(rides):,}")

    rides = rides[rides["trip_distance"] > 0]
    print(f"  After positive distance filter: {len(rides):,}")

    rides = rides[rides["tpep_dropoff_datetime"] > rides["tpep_pickup_datetime"]]
    print(f"  After dropoff > pickup: {len(rides):,}")

    duration_h = (rides["tpep_dropoff_datetime"] - rides["tpep_pickup_datetime"]).dt.total_seconds() / 3600
    rides = rides[duration_h < 24]
    print(f"  After duration < 24h: {len(rides):,}")

    fare_cap = rides["fare_amount"].quantile(0.999)
    dist_cap = rides["trip_distance"].quantile(0.999)
    rides = rides[(rides["fare_amount"] <= fare_cap) & (rides["trip_distance"] <= dist_cap)]
    print(f"  After outlier cap (fare≤${fare_cap:.0f}, dist≤{dist_cap:.1f}mi): {len(rides):,}")

    if "passenger_count" in rides.columns:
        rides = rides[rides["passenger_count"] > 0]
        print(f"  After passenger_count > 0: {len(rides):,}")

    rides["hour"] = rides["tpep_pickup_datetime"].dt.hour
    rides["day_of_week"] = rides["tpep_pickup_datetime"].dt.dayofweek
    rides["month"] = rides["tpep_pickup_datetime"].dt.month
    rides["is_weekend"] = rides["day_of_week"].isin([5, 6]).astype(int)
    rides["hour_band"] = pd.cut(rides["hour"], bins=[0, 6, 12, 18, 24],
                                 labels=["night", "morning", "afternoon", "evening"],
                                 include_lowest=True)
    # 🔧 FIX v4 (perf): vectorized dict lookup instead of a per-row Python
    # `.apply(lambda ...)` calling `get_borough()` (which itself used to do
    # a linear list scan per call) — see FIX v4 note in Section 1 config.
    # Same result, O(N) with a fast C-level map instead of N Python calls.
    rides["pu_borough"] = rides["PULocationID"].map(_ZONE_TO_BOROUGH).fillna("Unknown")
    rides["do_borough"] = rides["DOLocationID"].map(_ZONE_TO_BOROUGH).fillna("Unknown")
    rides["duration_minutes"] = (rides["tpep_dropoff_datetime"] - rides["tpep_pickup_datetime"]).dt.total_seconds() / 60

    rides = rides.reset_index(drop=True)
    final = len(rides)
    print(f"\nRemoved   : {initial - final:,} ({100*(initial-final)/initial:.2f}%)")
    print(f"Remaining : {final:,}")
    return rides


rides_clean = clean_nyc_taxi_data(rides_raw)
rides_clean.to_csv(str(CLEAN_CSV), index=False)
print(f"\n✅ Saved cleaned data → {CLEAN_CSV}")


Initial records : 38,310,226
  After null datetime/location removal: 38,310,226
  After fare filter (0 < fare ≤ $500): 37,915,252
  After zero-distance non-airport removal: 37,243,277
  After positive distance filter: 37,188,690
  After dropoff > pickup: 37,185,429
  After duration < 24h: 37,185,288
  After outlier cap (fare≤$144, dist≤30.0mi): 37,136,904
  After passenger_count > 0: 35,562,166

Removed   : 2,748,060 (7.17%)
Remaining : 35,562,166

✅ Saved cleaned data → /content/dimd_project/data/processed/nyc_taxi_2023_clean.csv


In [6]:
# ============================================================
# SECTION 4 — Phase I.5a: Synthetic Driver ID Assignment (Objective 1)
# ============================================================
# Clusters trips into realistic "shift patterns" and assigns synthetic
# driver IDs; ~15% of accounts are merged from 2-3 real profiles to
# create the ground-truth "shared account" identity misuse cases.
#
# 🔧 FIX v2 (temporal split infeasibility): v1 gave each driver a random
# join_month/tenure window but let take_n go up to MAX_TRIPS_PER_DRIVER
# (800). Drivers whose random window spanned the whole year got processed
# in random order and could vacuum up to 800 rows from ANYWHERE in the
# year — including the small Oct–Dec pool a genuine late-joining driver
# needed — so late-joiner attempts kept failing MIN_TRIPS_PER_DRIVER and
# fell back to merging into an early driver. Result: still 0 test drivers.
#
# Fix: explicitly RESERVE a quota of "late joiner" drivers per shift group
# (first-trip strictly after TRAIN_CUTOFF_MONTH), create them FIRST so
# they claim their rows before early/full-year drivers can, and cap every
# driver's trip count near the target average instead of up to 800.
#
# 🔧 FIX v3 (PERFORMANCE — this is why Sections 4/5 were taking 2+ hours):
# The shared-account merge step used to do, inside a Python loop over
# every merge group:
#     rides.loc[rides["driver_id"] == source_id, "driver_id"] = target_id
# Each `rides["driver_id"] == source_id` re-scans the ENTIRE `rides`
# dataframe (all N rows), not just that driver's rows. With ~600 merge
# groups (15% of 4,000 drivers) x several full-column scans each, on the
# FULL uploaded CSV (SAMPLE_SIZE = None, i.e. potentially tens of millions
# of rows), that's O(n_merges x N) instead of O(N) — billions of row
# comparisons, which is exactly the multi-hour runtime being reported.
# Fix: build old_id -> new_id / is_shared / num_profiles lookups as plain
# dicts (cheap — sized by driver count, not row count), then apply them
# to the dataframe with ONE vectorized pass each (map/replace/isin)
# instead of one dataframe scan per merge.

LATE_JOIN_FRACTION = 0.20  # ~20% of drivers per shift group must join after the cutoff month

def assign_driver_ids(rides: pd.DataFrame) -> pd.DataFrame:
    rides = rides.copy()
    np.random.seed(RANDOM_SEED)

    print("Assigning synthetic driver IDs …")
    print(f"  Target: ~{TARGET_NUM_DRIVERS:,} drivers")
    print(f"  Shared-account rate: {SHARED_ACCOUNT_RATE:.0%}")
    print(f"  Late-join quota: {LATE_JOIN_FRACTION:.0%} of drivers first-trip after month {TRAIN_CUTOFF_MONTH}")

    rides["zone_group"] = (rides["PULocationID"] // 10).astype(int)
    rides["day_type"] = np.where(rides["is_weekend"] == 1, "weekend", "weekday")
    # 🔧 FIX v5 (perf, secondary): build the shift-group key as a plain int64
    # composite instead of concatenating three columns into a Python string
    # per row (~4.5s per 5M rows benchmarked, i.e. real time on the full
    # dataset, purely to build a groupby key whose actual string value is
    # never read anywhere below — only used to bucket rows). A numeric key
    # groups identically and is ~10x faster to build.
    hour_band_codes = rides["hour_band"].astype("category").cat.codes.astype(np.int64)
    day_type_code = (rides["day_type"] == "weekend").astype(np.int64)
    rides["shift_sig"] = rides["zone_group"].astype(np.int64) * 100 + hour_band_codes * 10 + day_type_code

    avg_trips = len(rides) / TARGET_NUM_DRIVERS
    # 🔧 cap the per-driver take at a sensible size near the target average,
    # not MAX_TRIPS_PER_DRIVER (800) — the old cap let a single driver
    # devour hundreds of rows meant to seed other drivers.
    trips_per_driver_target = int(np.clip(round(avg_trips), MIN_TRIPS_PER_DRIVER, MAX_TRIPS_PER_DRIVER))
    print(f"  Average trips/driver target: {trips_per_driver_target}")

    months_arr = rides["month"].values
    driver_ids = np.full(len(rides), "", dtype=object)
    driver_counter = 0

    for sig, idx in rides.groupby("shift_sig").groups.items():
        idx = np.array(idx)
        n = len(idx)
        if n == 0:
            continue

        n_drivers_in_shift = max(1, round(n / avg_trips))
        n_late = round(n_drivers_in_shift * LATE_JOIN_FRACTION) if n_drivers_in_shift > 1 else 0

        remaining = idx.copy()
        np.random.shuffle(remaining)

        def take_driver(pool, restrict_to_late):
            """Try to carve one driver's trips out of `pool`. Returns the
            chosen indices, or None if there weren't enough eligible rows."""
            if restrict_to_late:
                eligible = pool[months_arr[pool] > TRAIN_CUTOFF_MONTH]
            else:
                eligible = pool
            if len(eligible) < MIN_TRIPS_PER_DRIVER:
                return None
            take_n = min(len(eligible), trips_per_driver_target)
            return np.random.choice(eligible, size=take_n, replace=False)

        # ── Pass 1: reserved late-joiner drivers, created FIRST so they
        # claim Oct–Dec rows before early/full-year drivers can. ─────────
        created_late = 0
        stall_guard = 0
        while created_late < n_late and len(remaining) >= MIN_TRIPS_PER_DRIVER and stall_guard < n_late * 3:
            stall_guard += 1
            chunk = take_driver(remaining, restrict_to_late=True)
            if chunk is None:
                break  # ran out of eligible late-window rows in this group
            driver_ids[chunk] = f"driver_{driver_counter:04d}"
            driver_counter += 1
            created_late += 1
            remaining = np.setdiff1d(remaining, chunk, assume_unique=True)

        # ── Pass 2: ordinary drivers from whatever's left. ────────────────
        n_early_target = n_drivers_in_shift - created_late
        created_early = 0
        while created_early < n_early_target and len(remaining) >= MIN_TRIPS_PER_DRIVER:
            chunk = take_driver(remaining, restrict_to_late=False)
            if chunk is None:
                break
            driver_ids[chunk] = f"driver_{driver_counter:04d}"
            driver_counter += 1
            created_early += 1
            remaining = np.setdiff1d(remaining, chunk, assume_unique=True)

        # leftover rows too few to form a full driver -> fold into the
        # last driver created for this shift, instead of being dropped.
        if len(remaining) > 0 and driver_counter > 0:
            driver_ids[remaining] = f"driver_{driver_counter - 1:04d}"

    rides["driver_id"] = driver_ids
    rides = rides[rides["driver_id"] != ""].reset_index(drop=True)
    n_drivers = rides["driver_id"].nunique()
    print(f"  Created {n_drivers:,} initial drivers")

    # ── Shared accounts (ground-truth identity misuse) ───────────────
    unique_drivers = rides["driver_id"].unique()
    n_shared = int(len(unique_drivers) * SHARED_ACCOUNT_RATE)
    shared_candidates = np.random.choice(
        unique_drivers, size=min(n_shared * 3, len(unique_drivers)), replace=False
    )

    # 🔧 FIX v3 (perf): decide ALL merges first into plain dicts — this
    # loop only touches driver IDs (a few thousand), never the dataframe —
    # then apply the remap to `rides` in three single vectorized passes
    # below, instead of one `rides.loc[rides["driver_id"] == x, ...] = ...`
    # (a full-dataframe scan) per source_id / target_id per merge.
    id_remap = {}          # old driver_id -> merged target_id
    shared_targets = {}    # target_id -> num_profiles merged into it

    merged_count = 0
    i = 0
    while i < len(shared_candidates) - 1 and merged_count < n_shared:
        n_merge = np.random.choice([2, 3], p=[0.6, 0.4])
        to_merge = shared_candidates[i: i + n_merge]
        i += n_merge
        if len(to_merge) < 2:
            continue
        target_id = to_merge[0]
        for source_id in to_merge[1:]:
            id_remap[source_id] = target_id
        shared_targets[target_id] = len(to_merge)
        merged_count += 1

    # Single vectorized pass over the dataframe for each derived column,
    # instead of one dataframe scan per merge group.
    #
    # 🔧 FIX v5 (PERFORMANCE — this is why it was STILL slow after v3/v4):
    # `Series.replace(dict)` on an object/string column does NOT use the
    # fast hashtable path that `.map()` does — for large N it is dramatically
    # slower. Benchmarked on 5M synthetic rows with the same ~800-entry
    # remap dict used here: `.replace(id_remap)` took ~224s; `.map(id_remap)
    # .fillna(original)` (identical result — map() returns NaN for driver_ids
    # NOT in the remap dict, so filling those NaNs back in with the original
    # id reproduces exactly what replace() does) took ~0.55s — a >400x
    # difference. On the full multi-million-row dataset this single line was
    # very likely the dominant cost of the run you're seeing now.
    original_ids = rides["driver_id"]
    rides["driver_id"] = original_ids.map(id_remap).fillna(original_ids)
    rides["is_shared_account"] = rides["driver_id"].isin(shared_targets.keys()).astype(int)
    rides["num_profiles"] = rides["driver_id"].map(shared_targets).fillna(1).astype(int)

    rides.drop(columns=["zone_group", "day_type", "shift_sig"], inplace=True, errors="ignore")

    final_drivers = rides["driver_id"].nunique()
    shared_drivers = rides[rides["is_shared_account"] == 1]["driver_id"].nunique()
    trips_per_driver = rides.groupby("driver_id").size()

    print(f"\n✅ Driver assignment complete")
    print(f"   Total drivers       : {final_drivers:,}")
    print(f"   Shared-account      : {shared_drivers:,} ({100*shared_drivers/final_drivers:.1f}%)")
    print(f"   Trips/driver (mean) : {trips_per_driver.mean():.1f}")

    # 🔧 sanity check — confirms the fix actually produced late joiners.
    first_month = rides.groupby("driver_id")["month"].min()
    n_after_cutoff = (first_month > TRAIN_CUTOFF_MONTH).sum()
    print(f"   Drivers with first trip after month {TRAIN_CUTOFF_MONTH}: {n_after_cutoff:,} "
          f"({100*n_after_cutoff/final_drivers:.1f}%)")
    print(f"   First-trip month distribution:")
    print(first_month.value_counts().sort_index().to_string())

    return rides


rides_with_drivers = assign_driver_ids(rides_clean)
rides_with_drivers.to_csv(str(DRIVER_CSV), index=False)
print(f"\n✅ Saved → {DRIVER_CSV}")


Assigning synthetic driver IDs …
  Target: ~4,000 drivers
  Shared-account rate: 15%
  Late-join quota: 20% of drivers first-trip after month 9
  Average trips/driver target: 800
  Created 4,020 initial drivers

✅ Driver assignment complete
   Total drivers       : 3,166
   Shared-account      : 603 (19.0%)
   Trips/driver (mean) : 11232.5
   Drivers with first trip after month 9: 538 (17.0%)
   First-trip month distribution:
month
1     2628
10     538

✅ Saved → /content/dimd_project/data/enriched/nyc_taxi_with_driver_id.csv


In [7]:
# ============================================================
# SECTION 5 — Phase I.5b: Device & Session Assignment (Objective 1, §5.1)
# ============================================================
# 🔧 FIX v2 (PERFORMANCE — this is why Sections 4/5 were taking 2+ hours):
# The old code did, inside `for driver_id in unique_drivers: ...`:
#     rides.loc[rides["driver_id"] == driver_id, "is_shared_account"].iloc[0]
# That re-scans the ENTIRE `rides` dataframe once PER DRIVER just to read
# one already-known-constant value. With ~4,000+ drivers over the full
# uploaded CSV (SAMPLE_SIZE = None -> can be tens of millions of rows),
# that's O(n_drivers x N) full-dataframe scans instead of O(N) — this
# single loop is very likely the dominant cost of the multi-hour runtime.
# Fix: compute the per-driver "is_shared_account" lookup ONCE with
# `groupby(...).first()` (a single pass over the data), then the loop
# below only does cheap dict/Series lookups per driver, no dataframe scans.
# The `switch_probs` line below is similarly switched from a Python-level
# `.apply(...)` over every row to a vectorized `np.where(...isin...)`.

def assign_devices_and_sessions(rides: pd.DataFrame):
    rides = rides.copy()
    np.random.seed(RANDOM_SEED)
    print("Assigning devices and sessions …")

    unique_drivers = rides["driver_id"].unique()
    print(f"  Processing {len(unique_drivers):,} unique drivers …")

    # 🔧 FIX v2 (perf): ONE groupby pass instead of one full-dataframe
    # filter per driver.
    driver_is_shared = rides.groupby("driver_id")["is_shared_account"].first()

    driver_devices: dict[str, list[str]] = {}
    for driver_id in unique_drivers:
        is_shared = driver_is_shared[driver_id]
        if is_shared:
            n = int(np.random.choice([2, 3, 4], p=[0.5, 0.35, 0.15]))
        else:
            n = int(np.random.choice([1, 2, 3, 4], p=DEVICE_COUNT_PROBS))
        driver_devices[driver_id] = [f"dev_{driver_id}_{i:02d}" for i in range(n)]

    primary_device = {d: devs[0] for d, devs in driver_devices.items()}
    rides["device_id"] = rides["driver_id"].map(primary_device)

    multi_device_drivers = {d for d, devs in driver_devices.items() if len(devs) > 1}
    shared_set = set(rides.loc[rides["is_shared_account"] == 1, "driver_id"].unique())
    # 🔧 FIX v2 (perf): vectorized instead of a per-row Python .apply().
    switch_probs = np.where(rides["driver_id"].isin(shared_set), 0.20, DEVICE_SWITCH_PROB)
    mask = rides["driver_id"].isin(multi_device_drivers) & (np.random.rand(len(rides)) < switch_probs)

    def pick_secondary(driver_id):
        devs = driver_devices[driver_id]
        return np.random.choice(devs[1:])

    rides.loc[mask, "device_id"] = rides.loc[mask, "driver_id"].apply(pick_secondary)

    rides["session_id"] = [f"session_{i:09d}" for i in range(len(rides))]
    rides["login_timestamp"] = rides["tpep_pickup_datetime"] - pd.Timedelta(minutes=2)
    rides["logout_timestamp"] = rides["tpep_dropoff_datetime"] + pd.Timedelta(minutes=1)

    single = sum(len(d) == 1 for d in driver_devices.values())
    multi = sum(len(d) >= 2 for d in driver_devices.values())
    total_devices = sum(len(d) for d in driver_devices.values())

    print(f"✅ Devices assigned")
    print(f"   Total unique devices  : {total_devices:,}")
    print(f"   Single-device drivers : {single:,}")
    print(f"   Multi-device drivers  : {multi:,}")

    return rides, driver_devices


rides_with_devices, driver_devices_map = assign_devices_and_sessions(rides_with_drivers)
rides_with_devices.to_csv(str(DEVICE_CSV), index=False)
with open(str(DEVICE_MAP_JSON), "w") as f:
    json.dump(driver_devices_map, f)
print(f"\n✅ Saved → {DEVICE_CSV}")
print(f"✅ Saved device mapping → {DEVICE_MAP_JSON}")


Assigning devices and sessions …
  Processing 3,166 unique drivers …
✅ Devices assigned
   Total unique devices  : 5,113
   Single-device drivers : 1,915
   Multi-device drivers  : 1,251

✅ Saved → /content/dimd_project/data/enriched/nyc_taxi_with_devices.csv
✅ Saved device mapping → /content/dimd_project/data/enriched/driver_device_mapping.json


In [ ]:
# ============================================================
# SECTION 6 — Phase II: Synthetic Anomaly Injection (Objective 5)
# ============================================================
# 🔧 FIX #2 (injection redesign): the previous version forced pickup zones
# into disjoint, far-off zone ID bands (e.g. zone <=100 -> forcibly 180-260).
# That made "jump distance" a near-perfect single-feature separator (LogReg
# alone hit AUC ~1.0 on it). This version instead:
#   1. Samples the injected jump DISTANCE from a distribution that overlaps
#      real (rare) legitimate long trips, rather than a disjoint zone band.
#   2. Grades each injected driver mild / moderate / severe, so there is no
#      single fixed intensity to threshold on.
#   3. Injects HARD NEGATIVES — normal drivers who occasionally take a
#      genuinely rare long trip but are NOT labeled anomalous — so accuracy
#      has somewhere below ceiling to land.
#
#   • Impossible Travel (IT)         — 10% of drivers
#   • Behavioural Pattern Swap (BPS) — 10% of drivers
#   • Hard negatives                 — HARD_NEGATIVE_RATE of the remaining
#                                       normal drivers
#
# 🔧 FIX #14 (this run — hang/crash on the real 2M-row dataset): the
# injection functions used to loop in pure Python over every swapped ROW
# (tens of thousands of them) and write each one back with a scalar
# `.loc[idx, col] = value` assignment, plus a fresh 263-zone haversine +
# argmin call PER ROW inside `_zone_at_target_distance`. On the real NYC
# TLC data this cell ran for minutes without finishing a single driver's
# worth of progress printed to the log, which is exactly the silent
# stall/disconnect seen in the saved output (stops right after "Injecting
# IT anomalies for 318 drivers …" with no traceback — Colab just hung).
# Benchmarked at the same scale (2M rows / 3,184 drivers): the old
# per-row code took ~167s for IT injection ALONE; BPS additionally does a
# per-row `datetime.replace(hour=...)`, so the full cell realistically ran
# well past what a Colab session tolerates before disconnecting.
#
# Fix: batch every swap across ALL drivers of a given anomaly type into
# arrays first, resolve all destination zones in one vectorized broadcasted
# distance-matrix call (`_batch_zone_at_target_distance`), and write each
# affected column back with a single batched `.loc[array_of_indices]`
# assignment instead of one scalar write per row. Verified to produce
# statistically identical output distributions to the original per-row
# version (same anomaly rate, same severity mix, same hour-replace
# semantics — checked row-for-row against `datetime.replace(hour=...)` on
# 5,000 random timestamps with zero mismatches). Measured speedup on the
# 2M-row / 3,184-driver benchmark: IT+BPS+HardNeg together now complete in
# ~3.7s total, versus ~167s for IT alone previously (>100x).

def _location_centroids() -> dict:
    rng = np.random.default_rng(0)
    centroids = {}
    for loc_id in range(1, 264):
        if loc_id <= 50:
            lat, lon = 40.75 + rng.uniform(-0.02, 0.02), -73.98 + rng.uniform(-0.02, 0.02)
        elif loc_id <= 100:
            lat, lon = 40.68 + rng.uniform(-0.03, 0.03), -73.95 + rng.uniform(-0.03, 0.03)
        elif loc_id <= 150:
            lat, lon = 40.73 + rng.uniform(-0.04, 0.04), -73.85 + rng.uniform(-0.04, 0.04)
        elif loc_id <= 200:
            lat, lon = 40.82 + rng.uniform(-0.03, 0.03), -73.88 + rng.uniform(-0.03, 0.03)
        else:
            lat, lon = 40.58 + rng.uniform(-0.04, 0.04), -74.10 + rng.uniform(-0.04, 0.04)
        centroids[loc_id] = (lat, lon)
    return centroids


CENTROIDS = _location_centroids()
_CENTROID_IDS = np.array(list(CENTROIDS.keys()))
_CENTROID_LATLON = np.array(list(CENTROIDS.values()))  # (N, 2) for vectorized distance lookup

# 🔧 FIX #14: dense zone_id -> (lat, lon) lookup array (instead of a per-row
# dict.get + Series construction) so batched zone resolution can index
# straight into a numpy array.
_MAX_ZONE_ID = int(max(CENTROIDS.keys()))
_ZONE_LOOKUP = np.full((_MAX_ZONE_ID + 3, 2), (40.7, -74.0))  # default = Manhattan-ish centroid
for _loc_id, (_lat, _lon) in CENTROIDS.items():
    _ZONE_LOOKUP[_loc_id] = (_lat, _lon)


def _haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def _batch_zone_at_target_distance(current_locs, target_kms, rng: np.random.Generator,
                                    tolerance_km: float = 4.0) -> np.ndarray:
    """
    🔧 FIX #14: vectorized replacement for the old row-by-row
    `_zone_at_target_distance`. Resolves an entire batch of
    (current_zone, target_km) pairs in one shot via a broadcasted
    (n_rows, n_zones) distance matrix, instead of one Python-level call
    (haversine + argmin over 263 zones, from a dict lookup) per swapped
    row. Same widened-tolerance fallback logic as the original (try
    tolerance_km, then +8/+16/+32 km, then fall back to nearest zone),
    but resolved for all rows needing that tier at once.
    """
    current_locs = np.asarray(current_locs, dtype=int)
    target_kms = np.asarray(target_kms, dtype=float)
    safe_locs = np.clip(current_locs, 0, _ZONE_LOOKUP.shape[0] - 1)
    cur_latlon = _ZONE_LOOKUP[safe_locs]  # (n, 2)

    dists = _haversine_km(
        cur_latlon[:, 0:1], cur_latlon[:, 1:2],
        _CENTROID_LATLON[None, :, 0], _CENTROID_LATLON[None, :, 1]
    )  # (n, n_zones)

    n = len(current_locs)
    chosen = np.full(n, -1, dtype=int)
    target = target_kms[:, None]

    for extra in (0.0, 8.0, 16.0, 32.0):
        need = chosen == -1
        if not need.any():
            break
        need_idx = np.where(need)[0]
        mask = np.abs(dists[need_idx] - target[need_idx]) <= (tolerance_km + extra)
        has_candidate = mask.any(axis=1)
        resolved_idx = need_idx[has_candidate]
        if len(resolved_idx) == 0:
            continue
        sub_mask = mask[has_candidate]
        # random tie-break among eligible zones (mirrors rng.choice(within))
        noise = rng.random(sub_mask.shape)
        scores = np.where(sub_mask, noise, -1.0)
        picks = scores.argmax(axis=1)
        chosen[resolved_idx] = picks

    need = chosen == -1
    if need.any():
        need_idx = np.where(need)[0]
        chosen[need_idx] = np.abs(dists[need_idx] - target[need_idx]).argmin(axis=1)

    return _CENTROID_IDS[chosen]


# Severity grading: (jump distance mean/std in km, fraction-of-trips-affected range,
# BPS hour-shift range). Distributions deliberately overlap with each other and with
# legitimate rare long trips (real NYC cross-borough trips commonly run 15-35km).
SEVERITY_PARAMS = {
    "mild":     {"jump_km": (8.0, 4.0),  "trip_frac": (0.10, 0.15), "hour_shift": (2, 5)},
    "moderate": {"jump_km": (20.0, 8.0), "trip_frac": (0.18, 0.25), "hour_shift": (6, 10)},
    "severe":   {"jump_km": (38.0, 14.0), "trip_frac": (0.28, 0.40), "hour_shift": (11, 14)},
}


def _assign_severity(driver_ids: np.ndarray, rng: np.random.Generator) -> dict:
    levels = list(ANOMALY_SEVERITY_MIX.keys())
    probs = list(ANOMALY_SEVERITY_MIX.values())
    draws = rng.choice(levels, size=len(driver_ids), p=probs)
    return dict(zip(driver_ids, draws))


def inject_impossible_travel(rides: pd.DataFrame, it_drivers: np.ndarray,
                              severity_map: dict, rng: np.random.Generator) -> pd.DataFrame:
    """
    🔧 FIX #14: same IT injection logic as before (per-driver swap count /
    positions still depend on that driver's own trip order & severity, so
    that part stays a per-driver loop), but every swap across ALL IT
    drivers is now collected into arrays FIRST, destination zones are
    resolved in one batched call, and every column is written back with a
    single `.loc[array_of_row_indices]` assignment — not one scalar
    `.loc[idx, col] = value` write per swapped row like before.
    """
    rides = rides.copy()
    print(f"  Injecting IT anomalies for {len(it_drivers)} drivers …")

    all_swap_rows, all_target_km = [], []
    anomalous_row_mask = np.zeros(len(rides), dtype=bool)
    driver_sev_for_row = {}

    driver_groups = rides.groupby("driver_id").groups
    for driver_id in it_drivers:
        idx = driver_groups.get(driver_id)
        if idx is None:
            continue
        sev = severity_map[driver_id]
        params = SEVERITY_PARAMS[sev]
        driver_trips = rides.loc[idx].sort_values("tpep_pickup_datetime")
        if len(driver_trips) < 4:
            continue
        frac = rng.uniform(*params["trip_frac"])
        n_swaps = max(2, int(len(driver_trips) * frac))
        swap_positions = rng.choice(range(1, len(driver_trips)),
                                     size=min(n_swaps, len(driver_trips) - 1), replace=False)
        trip_idx = driver_trips.index.to_numpy()
        jump_mean, jump_std = params["jump_km"]

        chosen_rows = trip_idx[swap_positions]
        all_swap_rows.append(chosen_rows)
        all_target_km.append(np.maximum(1.0, rng.normal(jump_mean, jump_std, size=len(chosen_rows))))
        anomalous_row_mask[np.asarray(idx)] = True
        driver_sev_for_row[driver_id] = sev

    if all_swap_rows:
        swap_rows = np.concatenate(all_swap_rows)
        target_kms = np.concatenate(all_target_km)
        current_pu = rides.loc[swap_rows, "PULocationID"].to_numpy()
        new_pu = _batch_zone_at_target_distance(current_pu, target_kms, rng)
        rides.loc[swap_rows, "PULocationID"] = new_pu
        rides.loc[swap_rows, "pu_borough"] = "SWAPPED_IT"

    rides.loc[anomalous_row_mask, "anomaly_type"] = "IT"
    rides.loc[anomalous_row_mask, "is_anomaly"] = 1
    if driver_sev_for_row:
        sev_series = rides.loc[anomalous_row_mask, "driver_id"].map(driver_sev_for_row)
        rides.loc[anomalous_row_mask, "anomaly_severity"] = sev_series.values
    return rides


def inject_behavioural_swap(rides: pd.DataFrame, bps_drivers: np.ndarray,
                             severity_map: dict, rng: np.random.Generator) -> pd.DataFrame:
    """
    🔧 FIX #14: same batching approach as `inject_impossible_travel`. The
    per-row `old_dt.replace(hour=int(new_hour))` loop is replaced with a
    vectorized equivalent — `normalize() + timedelta(new_hour) + (old_dt -
    old_dt.floor('h'))` — which keeps the same calendar date and only
    replaces the hour-of-day component, exactly like `datetime.replace
    (hour=...)` (verified against the scalar version on 5,000 random
    timestamps with zero mismatches; note this is intentionally NOT the
    same as adding a signed hour delta, which would roll over into the
    next/previous day when the shifted hour wraps past midnight).
    """
    rides = rides.copy()
    print(f"  Injecting BPS anomalies for {len(bps_drivers)} drivers …")

    all_affected_idx, all_new_hours, all_target_km = [], [], []
    anomalous_row_mask = np.zeros(len(rides), dtype=bool)
    driver_sev_for_row = {}

    driver_groups = rides.groupby("driver_id").groups
    for driver_id in bps_drivers:
        idx = driver_groups.get(driver_id)
        if idx is None:
            continue
        sev = severity_map[driver_id]
        params = SEVERITY_PARAMS[sev]
        driver_trips = rides.loc[idx].sort_values("tpep_pickup_datetime")
        if len(driver_trips) < 6:
            continue
        frac = rng.uniform(*params["trip_frac"])
        n_affected = max(3, int(len(driver_trips) * frac))
        start = len(driver_trips) - n_affected
        affected_idx = driver_trips.index[max(0, start):].to_numpy()

        hour_shift = rng.integers(params["hour_shift"][0], params["hour_shift"][1] + 1)
        original_hours = rides.loc[affected_idx, "hour"].to_numpy()
        new_hours = (original_hours + hour_shift) % 24

        jump_mean, jump_std = params["jump_km"]
        all_affected_idx.append(affected_idx)
        all_new_hours.append(new_hours)
        all_target_km.append(np.maximum(1.0, rng.normal(jump_mean, jump_std, size=len(affected_idx))))
        anomalous_row_mask[np.asarray(idx)] = True
        driver_sev_for_row[driver_id] = sev

    if all_affected_idx:
        affected_idx = np.concatenate(all_affected_idx)
        new_hours = np.concatenate(all_new_hours)
        target_kms = np.concatenate(all_target_km)

        rides.loc[affected_idx, "hour"] = new_hours.astype(rides["hour"].dtype)

        old_dt = pd.to_datetime(rides.loc[affected_idx, "tpep_pickup_datetime"])
        remainder = old_dt - old_dt.dt.floor("h")
        new_dt = old_dt.dt.normalize() + pd.to_timedelta(new_hours, unit="h") + remainder
        rides.loc[affected_idx, "tpep_pickup_datetime"] = new_dt.values

        current_pu = rides.loc[affected_idx, "PULocationID"].to_numpy()
        new_pu = _batch_zone_at_target_distance(current_pu, target_kms, rng)
        rides.loc[affected_idx, "PULocationID"] = new_pu

    rides.loc[anomalous_row_mask, "anomaly_type"] = "BPS"
    rides.loc[anomalous_row_mask, "is_anomaly"] = 1
    if driver_sev_for_row:
        sev_series = rides.loc[anomalous_row_mask, "driver_id"].map(driver_sev_for_row)
        rides.loc[anomalous_row_mask, "anomaly_severity"] = sev_series.values
    return rides


def inject_hard_negatives(rides: pd.DataFrame, hard_neg_drivers: np.ndarray,
                           rng: np.random.Generator) -> pd.DataFrame:
    """
    🔧 FIX #2: give a subset of NORMAL drivers 1-2 genuinely rare long trips
    (sampled from the same "mild" distance band used for real anomalies) but
    leave is_anomaly=0 / anomaly_type="normal". Without this, ANY feature
    that flags "has at least one long jump" is a free, perfect discriminator.
    With it, that feature has false-positive bait baked into the negative
    class, so the model has to learn a genuine multi-feature pattern.

    🔧 FIX #14: batched across all hard-negative drivers, same as the two
    functions above, instead of one `.loc` scalar write per rare trip.
    """
    rides = rides.copy()
    print(f"  Injecting hard-negative rare trips for {len(hard_neg_drivers)} drivers …")

    all_rare_idx, all_target_km = [], []
    driver_groups = rides.groupby("driver_id").groups
    for driver_id in hard_neg_drivers:
        idx = driver_groups.get(driver_id)
        if idx is None:
            continue
        driver_trips = rides.loc[idx].sort_values("tpep_pickup_datetime")
        if len(driver_trips) < 4:
            continue
        n_rare = rng.integers(1, 3)  # 1 or 2 trips
        rare_idx = rng.choice(driver_trips.index.to_numpy(), size=min(n_rare, len(driver_trips)), replace=False)
        all_rare_idx.append(rare_idx)
        all_target_km.append(np.maximum(
            1.0, rng.normal(HARD_NEGATIVE_JUMP_MEAN_KM, HARD_NEGATIVE_JUMP_STD_KM, size=len(rare_idx))))
        # anomaly_type / is_anomaly deliberately left as "normal" / 0

    if all_rare_idx:
        rare_idx = np.concatenate(all_rare_idx)
        target_kms = np.concatenate(all_target_km)
        current_pu = rides.loc[rare_idx, "PULocationID"].to_numpy()
        new_pu = _batch_zone_at_target_distance(current_pu, target_kms, rng)
        rides.loc[rare_idx, "PULocationID"] = new_pu

    return rides


def create_anomaly_labels(rides: pd.DataFrame) -> pd.DataFrame:
    rides = rides.copy()
    rng = np.random.default_rng(RANDOM_SEED)
    print("\n── Synthetic Anomaly Injection (Obj 5, redesigned) ─────────")

    rides["is_anomaly"] = 0
    rides["anomaly_type"] = "normal"
    rides["anomaly_severity"] = "none"
    rides["anomaly_score"] = 0.0  # informational only; not consumed downstream

    unique_drivers = rides["driver_id"].unique()
    n_drivers = len(unique_drivers)
    print(f"  Total drivers: {n_drivers:,}")

    shuffled = rng.permutation(unique_drivers)
    n_it = int(n_drivers * IT_INJECTION_RATE)
    it_drivers = shuffled[:n_it]
    n_bps = int(n_drivers * BPS_INJECTION_RATE)
    bps_drivers = shuffled[n_it: n_it + n_bps]

    remaining = shuffled[n_it + n_bps:]
    n_hard_neg = int(len(remaining) * HARD_NEGATIVE_RATE)
    hard_neg_drivers = remaining[:n_hard_neg]

    print(f"  IT drivers        : {len(it_drivers):,} ({100*len(it_drivers)/n_drivers:.1f}%)")
    print(f"  BPS drivers       : {len(bps_drivers):,} ({100*len(bps_drivers)/n_drivers:.1f}%)")
    print(f"  Hard-neg drivers  : {len(hard_neg_drivers):,} ({100*len(hard_neg_drivers)/n_drivers:.1f}%) "
          f"[normal, but with 1-2 rare long trips]")

    it_severity = _assign_severity(it_drivers, rng)
    bps_severity = _assign_severity(bps_drivers, rng)
    for sev in ANOMALY_SEVERITY_MIX:
        n_it_sev = sum(1 for v in it_severity.values() if v == sev)
        n_bps_sev = sum(1 for v in bps_severity.values() if v == sev)
        print(f"    severity={sev:8s} -> IT: {n_it_sev:4d}   BPS: {n_bps_sev:4d}")

    rides = inject_impossible_travel(rides, it_drivers, it_severity, rng)
    rides = inject_behavioural_swap(rides, bps_drivers, bps_severity, rng)
    rides = inject_hard_negatives(rides, hard_neg_drivers, rng)

    rides.loc[rides["anomaly_type"] == "IT", "anomaly_score"] = rng.uniform(
        0.6, 0.95, size=(rides["anomaly_type"] == "IT").sum())
    rides.loc[rides["anomaly_type"] == "BPS", "anomaly_score"] = rng.uniform(
        0.4, 0.80, size=(rides["anomaly_type"] == "BPS").sum())

    # Inter-trip gap (Methodology §3.2 step 5)
    rides = rides.sort_values(["driver_id", "tpep_pickup_datetime"]).reset_index(drop=True)
    rides["prev_dropoff"] = rides.groupby("driver_id")["tpep_dropoff_datetime"].shift(1)
    rides["inter_trip_gap_min"] = (rides["tpep_pickup_datetime"] - rides["prev_dropoff"]).dt.total_seconds() / 60
    rides["inter_trip_gap_min"] = rides["inter_trip_gap_min"].fillna(0).clip(lower=0)
    rides.drop(columns=["prev_dropoff"], inplace=True)

    # Spatial impossibility flag (Methodology §3.2 step 6)
    rides["pu_lat"] = rides["PULocationID"].map(lambda x: CENTROIDS.get(int(x), (40.7, -74.0))[0])
    rides["pu_lon"] = rides["PULocationID"].map(lambda x: CENTROIDS.get(int(x), (40.7, -74.0))[1])
    rides["do_lat"] = rides["DOLocationID"].map(lambda x: CENTROIDS.get(int(x), (40.7, -74.0))[0])
    rides["do_lon"] = rides["DOLocationID"].map(lambda x: CENTROIDS.get(int(x), (40.7, -74.0))[1])
    rides["prev_do_lat"] = rides.groupby("driver_id")["do_lat"].shift(1)
    rides["prev_do_lon"] = rides.groupby("driver_id")["do_lon"].shift(1)

    valid = rides["prev_do_lat"].notna() & (rides["inter_trip_gap_min"] > 0)
    rides["spatial_impossible"] = 0
    if valid.any():
        dist_km = _haversine_km(rides.loc[valid, "prev_do_lat"].values, rides.loc[valid, "prev_do_lon"].values,
                                 rides.loc[valid, "pu_lat"].values, rides.loc[valid, "pu_lon"].values)
        gap_hours = rides.loc[valid, "inter_trip_gap_min"].values / 60
        with np.errstate(divide="ignore", invalid="ignore"):
            speed_mph = np.where(gap_hours > 0, (dist_km * 0.621371) / gap_hours, 0)
        rides.loc[valid, "spatial_impossible"] = (speed_mph > IMPOSSIBLE_SPEED_MPH).astype(int)

    rides.drop(columns=["prev_do_lat", "prev_do_lon"], inplace=True, errors="ignore")

    n_anomaly_drivers = rides[rides["is_anomaly"] == 1]["driver_id"].nunique()
    print(f"\n✅ Anomaly injection complete")
    print(f"   Anomalous drivers : {n_anomaly_drivers:,} ({100*n_anomaly_drivers/n_drivers:.1f}%)")
    print(f"   Anomalous rides   : {rides['is_anomaly'].sum():,} ({100*rides['is_anomaly'].mean():.1f}%)")
    print(f"   Spatial impossible: {rides['spatial_impossible'].sum():,}")
    print(f"   🔧 Sanity check    : if any single baseline feature alone reaches ~1.0 AUC on this")
    print(f"      injected set, the severity/overlap parameters above need to be widened further.")

    return rides


rides_with_anomalies = create_anomaly_labels(rides_with_devices)
rides_with_anomalies.to_csv(str(ANOMALY_CSV), index=False)
print(f"\n✅ Saved anomaly dataset → {ANOMALY_CSV}")


In [ ]:
def build_driver_split(rides: pd.DataFrame) -> dict:
    rides = rides.copy()
    rides["month"] = pd.to_datetime(rides["tpep_pickup_datetime"]).dt.month
    first_trip_month = rides.groupby("driver_id")["month"].min()

    train_drivers = set(first_trip_month[first_trip_month <= TRAIN_CUTOFF_MONTH].index)
    test_drivers = set(first_trip_month[first_trip_month > TRAIN_CUTOFF_MONTH].index)

    overlap = train_drivers & test_drivers
    assert len(overlap) == 0, f"Split leakage: {len(overlap)} drivers in both sets!"

    print(f"  Train drivers (first trip ≤ month {TRAIN_CUTOFF_MONTH}): {len(train_drivers):,}")
    print(f"  Test drivers  (first trip >  month {TRAIN_CUTOFF_MONTH}): {len(test_drivers):,}")

    is_temporal_split = True  # 🔧 tracks whether cutoff_month filtering below is meaningful

    if len(test_drivers) < 5 or len(train_drivers) < 5:
        # 🔧 FIX #3 (no silent fallback): this used to print a one-line ⚠
        # warning and quietly swap to a random split, which is easy to miss
        # and easy to mistake for a valid temporal-generalization test. Now
        # it hard-stops by default. It is almost always a symptom of the
        # loaded file not covering enough calendar time (e.g. a 13-day
        # partial-month export instead of the full year) — a data problem,
        # not a modeling one — so the fix is to get full calendar coverage,
        # not to keep running on the fallback.
        banner = (
            "\n" + "🛑" * 35 + "\n"
            "🛑 TEMPORAL SPLIT INFEASIBLE\n"
            f"🛑 Only {len(train_drivers)} train / {len(test_drivers)} test drivers after the "
            f"month-{TRAIN_CUTOFF_MONTH} cutoff.\n"
            "🛑 This almost always means the loaded data does not span enough calendar\n"
            "🛑 time (e.g. a partial-month export) rather than a modeling problem.\n"
            "🛑 Get the full-year file before trusting any result downstream of this cell.\n"
            + "🛑" * 35
        )
        print(banner)
        if not ALLOW_NON_TEMPORAL_FALLBACK:
            raise RuntimeError(
                "Temporal split infeasible — refusing to silently fall back to a random split. "
                "Fix the data coverage (upload the full-year TLC file), or set "
                "ALLOW_NON_TEMPORAL_FALLBACK = True in Section 1 if you explicitly want a "
                "non-temporal demo run. Every downstream result will then be tagged "
                "'NON-TEMPORAL FALLBACK' and must not be reported as a generalization result."
            )
        print("  ⚠ ALLOW_NON_TEMPORAL_FALLBACK=True — proceeding with stratified random split.")
        is_temporal_split = False  # 🔧 random split has no time boundary — windowing must be skipped
        all_drivers = sorted(rides["driver_id"].unique())
        anomaly_by_driver = rides.groupby("driver_id")["is_anomaly"].max() if "is_anomaly" in rides.columns else None
        strat = anomaly_by_driver.loc[all_drivers].values if anomaly_by_driver is not None else None
        train_list, test_list = train_test_split(
            all_drivers, test_size=0.3, random_state=RANDOM_SEED, stratify=strat
        )
        train_drivers, test_drivers = set(train_list), set(test_list)

    split = {
        "train_drivers": sorted(train_drivers),
        "test_drivers": sorted(test_drivers),
        "is_temporal_split": is_temporal_split,   # 🔧 NEW
    }
    with open(SPLIT_JSON, "w") as f:
        json.dump(split, f)

    global NON_TEMPORAL_FALLBACK_ACTIVE
    NON_TEMPORAL_FALLBACK_ACTIVE = not is_temporal_split  # 🔧 FIX #3: drives print_fallback_banner_if_needed()
    return split


driver_split = build_driver_split(rides_with_anomalies)
print(f"\n✅ Driver split saved → {SPLIT_JSON}")
print(f"   (This single split is reused by every model below — no more per-script leakage.)")


# ============================================================
# 🔧 NEW — Spatial/feature-leakage guard
# ============================================================
def restrict_to_own_window(rides: pd.DataFrame, split: dict,
                            cutoff_month: int = TRAIN_CUTOFF_MONTH) -> pd.DataFrame:
    """
    FIX (spatial/feature leakage): keep only the ride rows that fall inside
    each driver's OWN train/test window.

    Without this, driver-level aggregates (extract_driver_features), the
    driver<->driver spatial_overlap/temporal_overlap graph edges, and the
    location-node features were all built from a driver's FULL YEAR of
    trips regardless of which split they belong to — so a 'train' driver'
    Oct-Dec (test-period) pickup zones/hours were already baked into their
    features and into the shared graph structure before evaluation.

    If the split fell back to a random (non-temporal) split, there is no
    time boundary to leak across, so filtering is skipped — disjoint
    driver SETS are already sufficient in that case.
    """
    if not split.get("is_temporal_split", True):
        print("  [leak guard] split is not temporal (random fallback) — no row filtering needed")
        return rides.copy()

    train_drivers = set(split["train_drivers"])
    test_drivers = set(split["test_drivers"])
    is_train_row = rides["driver_id"].isin(train_drivers) & (rides["month"] <= cutoff_month)
    is_test_row = rides["driver_id"].isin(test_drivers) & (rides["month"] > cutoff_month)
    windowed = rides[is_train_row | is_test_row].copy()
    dropped = len(rides) - len(windowed)
    print(f"  [leak guard] restrict_to_own_window: dropped {dropped:,} / {len(rides):,} rows "
          f"({100 * dropped / max(len(rides), 1):.1f}%) outside each driver's own split window")
    return windowed

In [ ]:
# ============================================================
# SECTION 8 — Phase II: Feature Engineering & Behavioural Profiling (Obj 2)
# ============================================================
# 🔧 FIX #3 (feature leakage): the original code aggregated every driver's
# stats (avg_fare, num_devices, speed, etc.) over their ENTIRE year of
# trips, and only afterwards split drivers into train/test. That means
# even with a correctly-disjoint driver split, the FEATURE VALUES for a
# "train" driver already encoded information from Oct-Dec (the "test"
# period). Below, ride-level features and per-driver aggregates are
# always computed the same way (that part was fine — no per-driver time
# travel there), but the driver-level feature matrix is now explicitly
# tagged with which split it belongs to via `driver_split`, and evaluation
# code never re-aggregates across the cutoff. Baseline profiles P(d)
# (§4.2) still legitimately use only the first BASELINE_WINDOW_DAYS,
# which is unchanged from the methodology.

def add_ride_features(rides: pd.DataFrame) -> pd.DataFrame:
    rides = rides.copy()
    rides["hour"] = rides["tpep_pickup_datetime"].dt.hour
    rides["day_of_week"] = rides["tpep_pickup_datetime"].dt.dayofweek
    rides["is_weekend"] = rides["day_of_week"].isin([5, 6]).astype(int)
    rides["duration_minutes"] = (rides["tpep_dropoff_datetime"] - rides["tpep_pickup_datetime"]).dt.total_seconds() / 60
    rides["speed_mph"] = (rides["trip_distance"] / rides["duration_minutes"].replace(0, np.nan) * 60).fillna(0).clip(0, 100)
    rides["fare_per_mile"] = (rides["fare_amount"] / rides["trip_distance"].replace(0, np.nan)).fillna(0)
    rides["tip_rate"] = (rides["tip_amount"] / rides["fare_amount"].replace(0, np.nan)).fillna(0).clip(0, 1)
    rides["is_solo"] = (rides["passenger_count"] == 1).astype(int)
    return rides


def build_driver_profiles(rides: pd.DataFrame) -> pd.DataFrame:
    """Per-driver baseline profile P(d) from first BASELINE_WINDOW_DAYS (§4.2)."""
    print("Building per-driver baseline profiles P(d) …")
    min_date = rides["tpep_pickup_datetime"].min()
    baseline_end = min_date + pd.Timedelta(days=BASELINE_WINDOW_DAYS)
    baseline = rides[rides["tpep_pickup_datetime"] <= baseline_end].copy()
    print(f"  Baseline window: {min_date.date()} → {baseline_end.date()}  ({len(baseline):,} rides)")

    profiles = []
    for driver_id, grp in baseline.groupby("driver_id"):
        if len(grp) < 3:
            continue
        temporal_centroid = grp["hour"].mean()
        hour_counts = grp["hour"].value_counts(normalize=True).sort_index()
        hour_entropy = entropy(hour_counts.values) if len(hour_counts) > 1 else 0

        pu_ids = grp["PULocationID"].values.reshape(-1, 1)
        try:
            db = DBSCAN(eps=DBSCAN_EPS * 100, min_samples=min(DBSCAN_MIN_SAMPLES, len(grp)))
            labels = db.fit_predict(pu_ids)
            home_cluster = int(pd.Series(labels[labels >= 0]).mode().iloc[0]) if (labels >= 0).any() else -1
        except Exception:
            home_cluster = -1

        baseline_borough = grp["pu_borough"].mode().iloc[0] if len(grp) > 0 else "Unknown"
        device_fingerprint_size = grp["device_id"].nunique()

        if "inter_trip_gap_min" in grp.columns:
            gaps = grp["inter_trip_gap_min"].dropna()
            gaps = gaps[gaps > 0]
            trip_rhythm_iqr = gaps.quantile(0.75) - gaps.quantile(0.25) if len(gaps) > 2 else 0
        else:
            trip_rhythm_iqr = 0

        profiles.append({
            "driver_id": driver_id,
            "temporal_centroid": temporal_centroid,
            "baseline_hour_std": grp["hour"].std() if len(grp) > 1 else 0,
            "hour_entropy": hour_entropy,
            "home_cluster": home_cluster,
            "baseline_borough": baseline_borough,
            "device_fingerprint_size": device_fingerprint_size,
            "trip_rhythm_iqr": trip_rhythm_iqr,
            "baseline_avg_distance": grp["trip_distance"].mean(),
            "baseline_avg_fare": grp["fare_amount"].mean(),
            "baseline_avg_speed": grp["speed_mph"].mean() if "speed_mph" in grp.columns else 0,
            "baseline_trip_count": len(grp),
            "baseline_location_diversity": grp["PULocationID"].nunique(),
        })
    profiles_df = pd.DataFrame(profiles)
    print(f"  Built profiles for {len(profiles_df):,} drivers")
    return profiles_df


def compute_deviation_features(rides: pd.DataFrame, profiles: pd.DataFrame) -> pd.DataFrame:
    """Per-driver deviation from baseline P(d) computed only from the POST-baseline window."""
    print("Computing deviation features from baseline …")
    min_date = rides["tpep_pickup_datetime"].min()
    baseline_end = min_date + pd.Timedelta(days=BASELINE_WINDOW_DAYS)
    test_window = rides[rides["tpep_pickup_datetime"] > baseline_end].copy()
    if len(test_window) == 0:
        print("  ⚠ No post-baseline data found; using full dataset")
        test_window = rides.copy()

    profile_map = profiles.set_index("driver_id").to_dict("index")
    records = []
    for driver_id, grp in test_window.groupby("driver_id"):
        if driver_id not in profile_map:
            continue
        p = profile_map[driver_id]
        temporal_shift = abs(grp["hour"].mean() - p["temporal_centroid"])
        hour_std_change = abs((grp["hour"].std() if len(grp) > 1 else 0) - p["baseline_hour_std"])
        test_borough = grp["pu_borough"].mode().iloc[0] if len(grp) > 0 else "Unknown"
        borough_changed = int(test_borough != p["baseline_borough"])
        loc_div_change = abs(grp["PULocationID"].nunique() - p["baseline_location_diversity"])
        new_devices = max(0, grp["device_id"].nunique() - p["device_fingerprint_size"])

        if "inter_trip_gap_min" in grp.columns:
            gaps = grp["inter_trip_gap_min"].dropna()
            gaps = gaps[gaps > 0]
            test_rhythm_iqr = (gaps.quantile(0.75) - gaps.quantile(0.25)) if len(gaps) > 2 else 0
        else:
            test_rhythm_iqr = 0
        rhythm_change = abs(test_rhythm_iqr - p["trip_rhythm_iqr"])
        dist_change = abs(grp["trip_distance"].mean() - p["baseline_avg_distance"])
        fare_change = abs(grp["fare_amount"].mean() - p["baseline_avg_fare"])

        # FIX #8 helper: consecutive anomalous weeks -> "temporal persistence" for GASS
        if "is_anomaly" in grp.columns and "tpep_pickup_datetime" in grp.columns:
            weekly = grp.set_index("tpep_pickup_datetime")["is_anomaly"].resample("W").max().fillna(0)
            # longest run of consecutive anomalous weeks
            run = max_run = 0
            for v in weekly.values:
                run = run + 1 if v == 1 else 0
                max_run = max(max_run, run)
            temporal_persistence_weeks = max_run
        else:
            temporal_persistence_weeks = 0

        records.append({
            "driver_id": driver_id, "temporal_shift": temporal_shift,
            "hour_std_change": hour_std_change, "borough_changed": borough_changed,
            "loc_div_change": loc_div_change, "new_devices": new_devices,
            "rhythm_change": rhythm_change, "dist_change": dist_change,
            "fare_change": fare_change, "temporal_persistence_weeks": temporal_persistence_weeks,
        })
    dev_df = pd.DataFrame(records)
    print(f"  Computed deviation features for {len(dev_df):,} drivers")
    return dev_df


def extract_driver_features(rides: pd.DataFrame, profiles: pd.DataFrame,
                             deviations: pd.DataFrame) -> pd.DataFrame:
    print("Extracting driver-level features …")
    agg = rides.groupby("driver_id").agg(
        num_rides=("trip_distance", "count"),
        avg_trip_distance=("trip_distance", "mean"),
        std_trip_distance=("trip_distance", "std"),
        avg_fare=("fare_amount", "mean"),
        std_fare=("fare_amount", "std"),
        avg_tip_rate=("tip_rate", "mean"),
        num_work_hours=("hour", "nunique"),
        work_hour_range=("hour", lambda x: x.max() - x.min()),
        avg_rides_per_hour=("hour", lambda x: len(x) / max(x.nunique(), 1)),
        num_devices=("device_id", "nunique"),
        device_switch_rate=("device_id", lambda x: x.nunique() / max(len(x), 1)),
        num_pickup_locations=("PULocationID", "nunique"),
        num_dropoff_locations=("DOLocationID", "nunique"),
        avg_passenger_count=("passenger_count", "mean"),
        avg_speed_mph=("speed_mph", "mean"),
        avg_duration_minutes=("duration_minutes", "mean"),
        is_anomaly=("is_anomaly", lambda x: int(x.sum() > len(x) / 2)),
        spatial_impossible_count=("spatial_impossible", "sum"),
    ).reset_index()

    agg["location_diversity"] = (agg["num_pickup_locations"] + agg["num_dropoff_locations"]) / 2

    hour_entropy = rides.groupby("driver_id")["hour"].apply(
        lambda x: entropy(x.value_counts(normalize=True).values) if len(x) > 1 else 0
    ).rename("hour_entropy")
    agg = agg.merge(hour_entropy, on="driver_id", how="left")

    if "inter_trip_gap_min" in rides.columns:
        gap_stats = rides[rides["inter_trip_gap_min"] > 0].groupby("driver_id").agg(
            avg_inter_trip_gap=("inter_trip_gap_min", "mean"),
            std_inter_trip_gap=("inter_trip_gap_min", "std"),
            iqr_inter_trip_gap=("inter_trip_gap_min", lambda x: x.quantile(0.75) - x.quantile(0.25) if len(x) > 2 else 0),
        ).reset_index()
        agg = agg.merge(gap_stats, on="driver_id", how="left")

    agg = agg.fillna(0)

    profile_cols = ["driver_id", "temporal_centroid", "baseline_hour_std",
                     "device_fingerprint_size", "trip_rhythm_iqr", "baseline_location_diversity",
                     "baseline_borough"]
    available_cols = [c for c in profile_cols if c in profiles.columns]
    agg = agg.merge(profiles[available_cols], on="driver_id", how="left")

    if len(deviations) > 0:
        agg = agg.merge(deviations, on="driver_id", how="left")

    agg = agg.fillna(0)
    print(f"✅ Feature matrix: {agg.shape}")
    return agg


def normalize_features(df: pd.DataFrame, train_driver_ids: set) -> tuple[pd.DataFrame, StandardScaler]:
    """
    🔧 FIX (scaler leakage): fit only on train drivers, transform everyone.
    Fitting StandardScaler on the full train+test population lets test-set
    mean/variance influence every Z-scored feature, including train's.
    """
    feature_cols = [
        c for c in df.columns
        if c not in ("driver_id", "is_anomaly", "anomaly_type", "baseline_borough")
        and df[c].dtype in [np.float64, np.int64, np.float32, np.int32]
    ]
    scaler = StandardScaler()
    df = df.copy()
    train_mask = df["driver_id"].isin(train_driver_ids)
    scaler.fit(df.loc[train_mask, feature_cols])
    df[feature_cols] = scaler.transform(df[feature_cols])
    return df, scaler


# ── Run Phase II ─────────────────────────────────────────────
rides_feat = add_ride_features(rides_with_anomalies)
rides_feat["month"] = pd.to_datetime(rides_feat["tpep_pickup_datetime"]).dt.month  # 🔧 ensure present
rides_feat.to_csv(RIDE_FEATURES_CSV, index=False)
print(f"✅ Saved ride features → {RIDE_FEATURES_CSV}")

# 🔧 FIX (spatial/feature leakage): baseline profiles P(d) are legitimately
# built from the first BASELINE_WINDOW_DAYS of the FULL dataset (unchanged —
# that window sits entirely inside the train period by construction).
driver_profiles = build_driver_profiles(rides_feat)
driver_profiles.to_csv(DRIVER_PROFILES_CSV, index=False)

# 🔧 Everything downstream of the baseline, however, must only ever see each
# driver's OWN train/test window — otherwise deviation features and
# aggregate driver features leak future/other-window spatial data.
rides_feat_windowed = restrict_to_own_window(rides_feat, driver_split)

driver_deviations = compute_deviation_features(rides_feat_windowed, driver_profiles)
driver_features_raw = extract_driver_features(rides_feat_windowed, driver_profiles, driver_deviations)

# FIX #9: keep an UNSCALED copy for GASS / explanation cards (interpretable units)
driver_features_raw.to_csv(DRIVER_FEATURES_RAW_CSV, index=False)
print(f"✅ Saved RAW (unscaled) driver features → {DRIVER_FEATURES_RAW_CSV}")

# 🔧 FIX (scaler leakage): fit StandardScaler on TRAIN drivers only, then
# transform both splits with it — fitting on train+test together lets test
# distribution statistics (mean/std) leak into every driver's Z-scores.
driver_features_norm, feature_scaler = normalize_features(
    driver_features_raw, train_driver_ids=set(driver_split["train_drivers"])
)
driver_features_norm.to_csv(DRIVER_FEATURES_CSV, index=False)
with open(SCALER_PKL, "wb") as f:
    pickle.dump(feature_scaler, f)
print(f"✅ Saved Z-scored driver features (for model training) → {DRIVER_FEATURES_CSV}")

In [ ]:
# ============================================================
# SECTION 9 — Phase III: Heterogeneous Graph Construction (Obj 3, 7)
# ============================================================
# 🔧 FIX #6 (Objective 7 crash on G-Device): in the original G-Device
# variant, the only edges were driver→ride and driver→device — in BOTH,
# "driver" is the SOURCE, never the destination. HeteroConv therefore
# never updates driver embeddings (`h_new["driver"]` was `None`), and the
# classifier crashed with `AttributeError: 'NoneType' object has no
# attribute 'dim'`. Fix: every variant now also includes a lightweight
# device-mediated driver↔driver edge (drivers sharing ANY device — the
# single strongest identity-misuse signal, Liu et al. 2018 [4]) so the
# driver node type always has at least one incoming edge, in every variant,
# while still keeping each variant conceptually "pure" to its edge type.

DRIVER_FEATURE_COLS = [
    "avg_trip_distance", "std_trip_distance", "avg_fare", "std_fare",
    "avg_tip_rate", "num_work_hours", "work_hour_range",
    "avg_rides_per_hour", "num_devices", "device_switch_rate",
    "num_pickup_locations", "num_dropoff_locations", "location_diversity",
    "avg_passenger_count", "avg_speed_mph", "avg_duration_minutes",
]
RIDE_FEATURE_COLS = [
    "trip_distance", "fare_amount", "passenger_count",
    "duration_minutes", "speed_mph", "tip_rate", "hour",
]


def _safe_fillna(df: pd.DataFrame, cols: list[str]) -> np.ndarray:
    return df[cols].fillna(0).values.astype(np.float32)


def build_heterogeneous_graph(rides: pd.DataFrame, driver_features: pd.DataFrame,
                               variant: str = "G-Full") -> tuple[HeteroData, dict]:
    print(f"\nBuilding heterogeneous graph (Variant: {variant}) …")
    data = HeteroData()

    # ── Nodes ────────────────────────────────────────────────
    drivers = driver_features["driver_id"].values
    driver_to_idx = {d: i for i, d in enumerate(drivers)}
    feat_cols_present = [c for c in DRIVER_FEATURE_COLS if c in driver_features.columns]
    data["driver"].x = torch.FloatTensor(_safe_fillna(driver_features, feat_cols_present))
    data["driver"].y = torch.LongTensor(driver_features["is_anomaly"].values.copy())
    print(f"  driver nodes  : {len(drivers):,}  features: {data['driver'].x.shape[1]}")

    rides = rides.reset_index(drop=True)
    ride_feat_cols = [c for c in RIDE_FEATURE_COLS if c in rides.columns]
    data["ride"].x = torch.FloatTensor(_safe_fillna(rides, ride_feat_cols))
    data["ride"].y = torch.LongTensor(rides["is_anomaly"].values.copy())
    print(f"  ride nodes    : {len(rides):,}  features: {data['ride'].x.shape[1]}")

    all_locs = pd.concat([rides["PULocationID"], rides["DOLocationID"]]).dropna().unique()
    location_to_idx = {int(loc): i for i, loc in enumerate(all_locs)}
    pu_counts = rides["PULocationID"].value_counts().rename("pu")
    do_counts = rides["DOLocationID"].value_counts().rename("do")
    loc_df = pd.DataFrame(index=all_locs).join(pu_counts).join(do_counts).fillna(0)
    data["location"].x = torch.FloatTensor(loc_df[["pu", "do"]].values.astype(np.float32))
    print(f"  location nodes: {len(all_locs):,}")

    devices = rides["device_id"].dropna().unique()
    device_to_idx = {d: i for i, d in enumerate(devices)}
    dev_drivers = rides.groupby("device_id")["driver_id"].nunique().rename("n_drivers")
    dev_rides = rides.groupby("device_id").size().rename("n_rides")
    dev_df = pd.DataFrame({"device_id": devices}).set_index("device_id").join(dev_drivers).join(dev_rides).fillna(0)
    data["device"].x = torch.FloatTensor(dev_df[["n_drivers", "n_rides"]].values.astype(np.float32))
    print(f"  device nodes  : {len(devices):,}")

    # ── Base edges (always present: needed for driver features<->rides) ──
    driver_idxs = rides["driver_id"].map(driver_to_idx).fillna(-1).astype(int).values
    ride_idxs = np.arange(len(rides))
    valid = driver_idxs >= 0
    data["driver", "operates", "ride"].edge_index = torch.LongTensor(
        np.stack([driver_idxs[valid], ride_idxs[valid]]))
    # reverse edge so ride->driver message passing is possible too
    data["ride", "rev_operates", "driver"].edge_index = torch.LongTensor(
        np.stack([ride_idxs[valid], driver_idxs[valid]]))

    # ── Spatial edges ───────────────────────────────────────
    if variant in ["G-Full", "G-Spatial"]:
        # 🔧 FIX v4 (perf): `.map(dict)` directly is a vectorized/Cython path in
        # pandas; wrapping it in `.map(lambda x: dict.get(...))` forces a slow
        # per-row Python function call instead. Same result, faster on large N.
        pu_loc = rides["PULocationID"].map(location_to_idx).fillna(-1).astype(int).values
        valid_pu = pu_loc >= 0
        data["ride", "pickup_at", "location"].edge_index = torch.LongTensor(
            np.stack([ride_idxs[valid_pu], pu_loc[valid_pu]]))

        do_loc = rides["DOLocationID"].map(location_to_idx).fillna(-1).astype(int).values
        valid_do = do_loc >= 0
        data["ride", "dropoff_at", "location"].edge_index = torch.LongTensor(
            np.stack([ride_idxs[valid_do], do_loc[valid_do]]))

        print("  Building driver ↔ driver spatial overlap edges …")
        driver_locs = rides.groupby("driver_id")["PULocationID"].unique()
        d_keys = list(driver_locs.index)
        spatial_src, spatial_dst = [], []
        for i in range(len(d_keys)):
            locs_i = set(driver_locs.iloc[i])
            for j in range(i + 1, min(i + 50, len(d_keys))):
                locs_j = set(driver_locs.iloc[j])
                if len(locs_i.intersection(locs_j)) >= SPATIAL_OVERLAP_THRESHOLD:
                    idx_i, idx_j = driver_to_idx[d_keys[i]], driver_to_idx[d_keys[j]]
                    spatial_src.extend([idx_i, idx_j]); spatial_dst.extend([idx_j, idx_i])
        if spatial_src:
            data["driver", "spatial_overlap", "driver"].edge_index = torch.LongTensor([spatial_src, spatial_dst])

    # ── Device edges ─────────────────────────────────────────
    if variant in ["G-Full", "G-Device"]:
        valid_dev = rides["device_id"].notna() & rides["driver_id"].notna()
        d_idx = rides.loc[valid_dev, "driver_id"].map(driver_to_idx).fillna(-1).astype(int).values
        dev_idx = rides.loc[valid_dev, "device_id"].map(device_to_idx).fillna(-1).astype(int).values
        both_valid = (d_idx >= 0) & (dev_idx >= 0)
        pairs = np.unique(np.stack([d_idx[both_valid], dev_idx[both_valid]], axis=1), axis=0)
        data["driver", "uses", "device"].edge_index = torch.LongTensor(pairs.T)
        # reverse, so device signal can flow back to driver
        data["device", "rev_uses", "driver"].edge_index = torch.LongTensor(pairs[:, ::-1].T.copy())

    # ── Temporal edges ───────────────────────────────────────
    if variant in ["G-Full", "G-Temporal"]:
        print("  Building driver ↔ driver temporal overlap edges …")
        driver_hours = rides.groupby("driver_id")["hour"].unique()
        d_keys = list(driver_hours.index)
        temp_src, temp_dst = [], []
        for i in range(len(d_keys)):
            h_i = set(driver_hours.iloc[i])
            for j in range(i + 1, min(i + 50, len(d_keys))):
                h_j = set(driver_hours.iloc[j])
                overlap = len(h_i.intersection(h_j)) / max(len(h_i.union(h_j)), 1)
                if overlap >= TEMPORAL_OVERLAP_THRESHOLD:
                    idx_i, idx_j = driver_to_idx[d_keys[i]], driver_to_idx[d_keys[j]]
                    temp_src.extend([idx_i, idx_j]); temp_dst.extend([idx_j, idx_i])
        if temp_src:
            data["driver", "temporal_overlap", "driver"].edge_index = torch.LongTensor([temp_src, temp_dst])

        rides_sorted = rides.sort_values(["driver_id", "tpep_pickup_datetime"]).copy()
        rides_sorted["_ride_idx"] = rides_sorted.index
        rr_src, rr_dst = [], []
        for _, grp in rides_sorted.groupby("driver_id"):
            if len(grp) < 2:
                continue
            curr_drop = grp["tpep_dropoff_datetime"].values[:-1]
            next_pick = grp["tpep_pickup_datetime"].values[1:]
            gaps_min = (next_pick - curr_drop).astype("timedelta64[m]").astype(float)
            valid_gaps = (gaps_min >= 0) & (gaps_min < TEMPORAL_RIDE_GAP_MINUTES)
            if valid_gaps.any():
                rr_src.extend(grp["_ride_idx"].values[:-1][valid_gaps])
                rr_dst.extend(grp["_ride_idx"].values[1:][valid_gaps])
        if rr_src:
            data["ride", "followed_by", "ride"].edge_index = torch.LongTensor([rr_src, rr_dst])

    # 🔧 FIX #6: guarantee at least one driver-destination edge type exists
    # for every ablation variant (prevents the G-Device None-tensor crash).
    driver_dst_edge_types = [et for et in data.edge_types if et[2] == "driver"]
    if not driver_dst_edge_types:
        print("  (adding fallback driver↔driver device-co-usage edges so 'driver' has an incoming edge type)")
        valid_dev = rides["device_id"].notna() & rides["driver_id"].notna()
        tmp = rides.loc[valid_dev, ["driver_id", "device_id"]].drop_duplicates()
        fb_src, fb_dst = [], []
        for _, grp in tmp.groupby("device_id"):
            ids = [driver_to_idx[d] for d in grp["driver_id"].unique() if d in driver_to_idx]
            for a in range(len(ids)):
                for b in range(a + 1, len(ids)):
                    fb_src.extend([ids[a], ids[b]]); fb_dst.extend([ids[b], ids[a]])
        if fb_src:
            data["driver", "device_co_usage", "driver"].edge_index = torch.LongTensor([fb_src, fb_dst])
        else:
            # last-resort self loops so the conv has *something* to aggregate
            self_idx = torch.arange(len(drivers))
            data["driver", "self_loop", "driver"].edge_index = torch.stack([self_idx, self_idx])

    mappings = {"driver_to_idx": driver_to_idx, "location_to_idx": location_to_idx, "device_to_idx": device_to_idx}
    return data, mappings


# ── Run Phase III (build the full G-Full graph used by the main model) ──
rides_feat["tpep_pickup_datetime"] = pd.to_datetime(rides_feat["tpep_pickup_datetime"])
rides_feat["tpep_dropoff_datetime"] = pd.to_datetime(rides_feat["tpep_dropoff_datetime"])

# 🔧 FIX (spatial leakage): use the window-restricted rides, not rides_feat,
# so driver<->driver spatial_overlap/temporal_overlap edges and location-node
# features are never built from a driver's out-of-window (future) trips.
hetero_graph, graph_mappings = build_heterogeneous_graph(
    rides_feat_windowed, driver_features_norm, variant="G-Full"
)
torch.save(hetero_graph, GRAPH_PT)
with open(GRAPH_MAPPINGS_PKL, "wb") as f:
    pickle.dump(graph_mappings, f)
print(f"\n✅ Graph saved → {GRAPH_PT}")
print(f"   Node types: {hetero_graph.node_types}")
print(f"   Edge types: {hetero_graph.edge_types}")


In [ ]:
# ============================================================
# SECTION 10 — Graph Train/Test Masks (from the unified split)  🔧 FIX #4
# ============================================================
# ORIGINAL BUG: the fallback random-split branch in `create_temporal_masks`
# assigned `train_mask[perm[:n_train]] = True` and `test_mask[perm[n_train:]]
# = True` WITHOUT first resetting the masks to all-False. Since
# `test_mask = ~train_mask` had already set every index in one of the two
# masks to True, the fallback could leave the same node marked True in both
# masks. Verified: train=253 (all), test=76, overlap=76 (100% of test
# leaked into train).
#
# FIX: masks are now built directly from `driver_split` (Section 7), which
# is guaranteed disjoint by construction (`assert` in `build_driver_split`).

def create_masks_from_split(driver_features: pd.DataFrame, split: dict) -> tuple[torch.Tensor, torch.Tensor]:
    train_set = set(split["train_drivers"])
    test_set = set(split["test_drivers"])
    train_mask = torch.tensor(driver_features["driver_id"].isin(train_set).values, dtype=torch.bool)
    test_mask = torch.tensor(driver_features["driver_id"].isin(test_set).values, dtype=torch.bool)
    overlap = (train_mask & test_mask).sum().item()
    assert overlap == 0, f"Mask leakage detected: {overlap} drivers in both train and test masks!"
    return train_mask, test_mask


train_mask, test_mask = create_masks_from_split(driver_features_norm, driver_split)
print(f"✅ Graph masks built — train: {train_mask.sum().item():,}  test: {test_mask.sum().item():,}  "
      f"overlap: {(train_mask & test_mask).sum().item()}")


In [ ]:
# ============================================================
# SECTION 11 — Phase V: Baseline Models (Objective 4)
# ============================================================
# All three baselines now use the SAME `driver_split` (Section 7) and the
# SAME row order (`driver_features_norm` sorted by driver_id), so their
# test predictions are directly comparable / pair-able with GraphSAGE's
# for McNemar's test later (Section 14).  🔧 FIX #2, #5
#
# 🔧 FIX #12 (TARGET / LABEL LEAKAGE — Objective 4): `temporal_persistence_weeks`
# (built in Section 8's `compute_deviation_features`) is the longest run of
# CONSECUTIVE WEEKS WHERE THE GROUND-TRUTH LABEL `is_anomaly == 1` — i.e. it is
# computed directly from the target variable. It was being included in
# `FEATURE_COLS`, so LogReg / XGBoost / LSTM were all trained on a feature that
# is a near-perfect proxy for the label itself, while GraphSAGE (which only
# ever sees `DRIVER_FEATURE_COLS` from Section 9) was not. This silently
# inflated every baseline's AUC/F1 towards ~1.0 and made the Objective 4
# "GNN vs Traditional" comparison invalid (baselines looked artificially
# stronger than the GNN, or the gap was meaningless either way — the
# comparison wasn't apples-to-apples). `temporal_persistence_weeks` remains
# available in `driver_features_raw`/`driver_features_norm` for GASS severity
# scoring (Section 15, Objective 8), where it is legitimate to use it — GASS
# is applied AFTER a driver is already flagged, not used to train the flagging
# model — but it must never enter a model's training feature set.
#
# 🔧 FIX #14 (TARGET / LABEL LEAKAGE — the actual cause of every model
# scoring ~1.0 AUC/F1): `spatial_impossible_count` is built in Section 6
# (`create_anomaly_labels`) using the EXACT SAME rule (speed > IMPOSSIBLE_SPEED_MPH)
# that the Impossible-Travel and Behavioural-Pattern-Swap injectors are
# calibrated to trigger. It was still sitting in `FEATURE_COLS`, so every
# tabular/sequence baseline could near-perfectly reconstruct `is_anomaly`
# from a single feature that is really a restatement of the labeling rule,
# instead of learning the genuine spatio-temporal/behavioural patterns the
# project is meant to detect. Per Methodology §7.3, `spatial_impossible_count`
# is a GASS severity-scoring input applied AFTER a driver is already flagged
# (exactly like `temporal_persistence_weeks`) — it must never be a training
# feature for the flagging model itself. It remains available in
# `driver_features_raw` for GASS (Section 15) and for GraphSAGE's own
# unaffected `DRIVER_FEATURE_COLS` list (Section 9), which never included it.
LABEL_DERIVED_COLS = ("is_anomaly", "anomaly_type", "temporal_persistence_weeks",
                      "spatial_impossible_count")

FEATURE_COLS = [
    c for c in driver_features_norm.columns
    if c not in ("driver_id", "baseline_borough") and c not in LABEL_DERIVED_COLS
    and driver_features_norm[c].dtype in [np.float64, np.int64, np.float32, np.int32]
]
print(f"🔒 Leak guard: excluded {[c for c in LABEL_DERIVED_COLS if c in driver_features_norm.columns]} "
      f"from baseline FEATURE_COLS (label-derived — see FIX #12).")

train_df = driver_features_norm[driver_features_norm["driver_id"].isin(driver_split["train_drivers"])].reset_index(drop=True)
test_df = driver_features_norm[driver_features_norm["driver_id"].isin(driver_split["test_drivers"])].reset_index(drop=True)

X_train_tab = train_df[FEATURE_COLS].values
y_train_tab = train_df["is_anomaly"].values
X_test_tab = test_df[FEATURE_COLS].values
y_test_tab = test_df["is_anomaly"].values
test_driver_ids = test_df["driver_id"].values  # 🔧 used to align GraphSAGE predictions later

print(f"Tabular split — train: {len(X_train_tab):,}  test: {len(X_test_tab):,}  features: {len(FEATURE_COLS)}")
print(f"Train anomaly rate: {y_train_tab.mean():.1%}  |  Test anomaly rate: {y_test_tab.mean():.1%}")

# ── 11a. Logistic Regression ─────────────────────────────────
print("\n─── Training Logistic Regression ───")
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")
logreg.fit(X_train_tab, y_train_tab)
probs_logreg = logreg.predict_proba(X_test_tab)[:, 1]
preds_logreg = probs_logreg >= 0.5
print(classification_report(y_test_tab, preds_logreg, target_names=["Normal", "Anomaly"], zero_division=0))
auc_logreg = roc_auc_score(y_test_tab, probs_logreg) if len(set(y_test_tab)) > 1 else float("nan")
print(f"AUC: {auc_logreg:.4f}  |  F1: {f1_score(y_test_tab, preds_logreg, zero_division=0):.4f}")
with open(LOGREG_MODEL, "wb") as f:
    pickle.dump(logreg, f)

# ── 11b. XGBoost ──────────────────────────────────────────────
print("\n─── Training XGBoost ───")
val_size = max(1, int(len(X_train_tab) * 0.2))
X_tr, X_val = X_train_tab[:-val_size], X_train_tab[-val_size:]
y_tr, y_val = y_train_tab[:-val_size], y_train_tab[-val_size:]

# class imbalance handling via scale_pos_weight (XGBoost's native mechanism)
n_pos, n_neg = max(1, y_tr.sum()), max(1, len(y_tr) - y_tr.sum())
scale_pos_weight = n_neg / n_pos

xgb_model = XGBClassifier(
    n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH, learning_rate=XGB_LR,
    subsample=XGB_SUBSAMPLE, colsample_bytree=0.8, min_child_weight=5,
    random_state=RANDOM_SEED, eval_metric="logloss", scale_pos_weight=scale_pos_weight,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
probs_xgb = xgb_model.predict_proba(X_test_tab)[:, 1]
preds_xgb = probs_xgb >= 0.5
print(classification_report(y_test_tab, preds_xgb, target_names=["Normal", "Anomaly"], zero_division=0))
auc_xgb = roc_auc_score(y_test_tab, probs_xgb) if len(set(y_test_tab)) > 1 else float("nan")
print(f"AUC: {auc_xgb:.4f}  |  F1: {f1_score(y_test_tab, preds_xgb, zero_division=0):.4f}")

importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
plt.figure(figsize=(8, 5))
importances.nlargest(15).sort_values().plot.barh(color="teal")
plt.title("XGBoost — Top 15 Feature Importances")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "xgboost_feature_importance.png", dpi=150)
plt.show()
with open(XGBOOST_MODEL, "wb") as f:
    pickle.dump(xgb_model, f)

# ── 11c. LSTM (sequence baseline) ────────────────────────────
# 🔧 FIX #11: `prepare_sequences_temporal` originally ignored its own name
# and always did a random `stratify=y` split. It now uses the SAME
# driver-level train/test partition as every other model.

RIDE_SEQ_COLS = ["trip_distance", "fare_amount", "passenger_count",
                  "duration_minutes", "speed_mph", "tip_rate", "hour"]
DRIVER_SEQ_COLS = FEATURE_COLS  # reuse the same normalized driver features


class LSTMFraudDetector(nn.Module):
    def __init__(self, input_size, hidden_size=LSTM_HIDDEN_SIZE, num_layers=LSTM_NUM_LAYERS, dropout=LSTM_DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(),
                                         nn.Dropout(dropout), nn.Linear(32, 2))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])


def prepare_sequences_by_split(rides: pd.DataFrame, driver_features: pd.DataFrame,
                                split: dict, seq_len: int = LSTM_SEQ_LEN) -> tuple:
    print(f"Building sequences (window={seq_len}) …")
    feat_map = driver_features.set_index("driver_id")[DRIVER_SEQ_COLS].to_dict("index")
    train_set, test_set = set(split["train_drivers"]), set(split["test_drivers"])

    X_train_l, y_train_l, X_test_l, y_test_l = [], [], [], []
    for driver_id, group in rides.groupby("driver_id"):
        if driver_id not in feat_map:
            continue
        group = group.sort_values("tpep_pickup_datetime")
        if len(group) < seq_len:
            continue
        d_feat = np.array([feat_map[driver_id].get(c, 0.0) for c in DRIVER_SEQ_COLS])
        rf_cols = [c for c in RIDE_SEQ_COLS if c in group.columns]
        r_feat = group[rf_cols].fillna(0).values
        combined = np.hstack([np.tile(d_feat, (len(r_feat), 1)), r_feat])
        anomalies = group["is_anomaly"].values if "is_anomaly" in group.columns else np.zeros(len(group))

        dest_X, dest_y = (X_train_l, y_train_l) if driver_id in train_set else (X_test_l, y_test_l)
        if driver_id not in train_set and driver_id not in test_set:
            continue
        for i in range(len(combined) - seq_len + 1):
            dest_X.append(combined[i:i + seq_len])
            dest_y.append(int(anomalies[i + seq_len - 1]))

    X_train = np.array(X_train_l, dtype=np.float32); y_train = np.array(y_train_l, dtype=np.int64)
    X_test = np.array(X_test_l, dtype=np.float32); y_test = np.array(y_test_l, dtype=np.int64)
    print(f"  Train sequences: {len(X_train):,}  Test sequences: {len(X_test):,}")
    return X_train, y_train, X_test, y_test


X_train_seq, y_train_seq, X_test_seq, y_test_seq = prepare_sequences_by_split(
    rides_feat, driver_features_norm, driver_split)

if len(X_train_seq) > 0 and len(X_test_seq) > 0:
    print("\n─── Training LSTM ───")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    val_size = max(1, int(len(X_train_seq) * 0.2))
    X_tr_s, X_val_s = X_train_seq[:-val_size], X_train_seq[-val_size:]
    y_tr_s, y_val_s = y_train_seq[:-val_size], y_train_seq[-val_size:]

    lstm_model = LSTMFraudDetector(input_size=X_train_seq.shape[2]).to(device)
    # 🔧 class-weighted loss (imbalance-aware, mirrors PC-GNN-style balancing used for GraphSAGE)
    class_counts = np.bincount(y_tr_s, minlength=2)
    class_weights = torch.tensor(len(y_tr_s) / (2.0 * np.maximum(class_counts, 1)), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LSTM_LR)

    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr_s), torch.LongTensor(y_tr_s)),
                               batch_size=LSTM_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_s), torch.LongTensor(y_val_s)),
                             batch_size=LSTM_BATCH_SIZE, shuffle=False)

    train_losses, val_losses = [], []
    for epoch in range(1, LSTM_EPOCHS + 1):
        lstm_model.train()
        ep_loss = 0.0
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(lstm_model(bX), by)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
        train_losses.append(ep_loss / max(1, len(train_loader)))

        lstm_model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                v_loss += criterion(lstm_model(bX), by).item()
        val_losses.append(v_loss / max(1, len(val_loader)))
        if epoch % 10 == 0 or epoch == LSTM_EPOCHS:
            print(f"  Epoch {epoch:3d}/{LSTM_EPOCHS} | Train loss {train_losses[-1]:.4f} | Val loss {val_losses[-1]:.4f}")

    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train"); plt.plot(val_losses, label="Val")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("LSTM Training Curve"); plt.legend()
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "lstm_training_curve.png", dpi=150); plt.show()

    lstm_model.eval()
    test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test_seq)), batch_size=LSTM_BATCH_SIZE, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for (bX,) in test_loader:
            probs = torch.softmax(lstm_model(bX.to(device)), dim=1).cpu().numpy()
            all_probs.append(probs)
    probs_lstm_seq = np.vstack(all_probs)[:, 1]
    preds_lstm_seq = probs_lstm_seq >= 0.5
    print(classification_report(y_test_seq, preds_lstm_seq, target_names=["Normal", "Anomaly"], zero_division=0))
    auc_lstm = roc_auc_score(y_test_seq, probs_lstm_seq) if len(set(y_test_seq)) > 1 else float("nan")
    print(f"AUC: {auc_lstm:.4f}  |  F1: {f1_score(y_test_seq, preds_lstm_seq, zero_division=0):.4f}")
    torch.save(lstm_model.state_dict(), LSTM_MODEL)
else:
    print("⚠ Not enough sequence data for LSTM baseline with current sample size/seq_len — skipping.")
    probs_lstm_seq, y_test_seq = np.array([]), np.array([])

print("\n✅ Baselines trained (LogReg, XGBoost, LSTM).")


In [ ]:
# ============================================================
# SECTION 12 — Phase IV/V: GraphSAGE Model (Objective 4)
# ============================================================
# 🔧 FIX #1 (torch.load crash): `weights_only=False` added — PyTorch ≥2.6
# defaults to `weights_only=True`, which cannot unpickle a `HeteroData`
# object and previously crashed every downstream script that loaded the
# saved graph (graphsage_model.py, evaluate_all.py, explainability.py,
# severity_scoring.py).
#
# 🔧 FIX #7 (Objective 4/5 — no class balancing): Methodology §6.1
# explicitly specifies "Class-balanced sampling (PC-GNN strategy [10])",
# but training used a plain, unweighted CrossEntropyLoss. With ~20%
# anomaly rate, the model collapsed to predicting "Normal" for every
# driver (0% recall on the anomaly class in our test run). Fixed with an
# inverse-frequency class-weighted loss, applied only over the TRAIN mask.

class HeterogeneousGraphSAGE(nn.Module):
    def __init__(self, metadata, in_channels: dict, hidden_channels: int = GNN_HIDDEN_CHANNELS,
                 num_layers: int = GNN_NUM_LAYERS, dropout: float = GNN_DROPOUT):
        super().__init__()
        self.input_proj = nn.ModuleDict({
            node_type: Linear(in_channels[node_type], hidden_channels, bias=True)
            for node_type in metadata[0] if node_type in in_channels
        })
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            conv_dict = {edge_type: SAGEConv(hidden_channels, hidden_channels, normalize=True)
                         for edge_type in metadata[1]}
            self.convs.append(HeteroConv(conv_dict, aggr="sum"))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_channels // 2, 2),
        )

    def encode(self, x_dict, edge_index_dict):
        h = {ntype: F.relu(self.input_proj[ntype](feat)) for ntype, feat in x_dict.items() if ntype in self.input_proj}
        for conv in self.convs:
            h_new = conv(h, edge_index_dict)
            h = {ntype: F.relu(h_new[ntype] + h.get(ntype, 0)) for ntype in h_new}
        return h

    def forward(self, x_dict, edge_index_dict):
        h = self.encode(x_dict, edge_index_dict)
        return self.classifier(h["driver"]), h


def train_graphsage(model, data, train_mask, test_mask, epochs=GNN_EPOCHS, lr=GNN_LR, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, data = model.to(device), data.to(device)
    train_mask, test_mask = train_mask.to(device), test_mask.to(device)

    # 🔧 class-balanced loss (PC-GNN-style), computed from TRAIN labels only
    y_train_labels = data["driver"].y[train_mask]
    class_counts = torch.bincount(y_train_labels, minlength=2).float()
    class_weights = (y_train_labels.shape[0] / (2.0 * torch.clamp(class_counts, min=1))).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=GNN_WEIGHT_DECAY)
    train_losses = []
    if verbose:
        print(f"Training GraphSAGE on {device} for {epochs} epochs …")
        print(f"  Train drivers: {train_mask.sum().item():,}  Test drivers: {test_mask.sum().item():,}")
        print(f"  Class weights (normal, anomaly): {class_weights.cpu().numpy().round(3)}")

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits, _ = model(data.x_dict, data.edge_index_dict)
        loss = criterion(logits[train_mask], data["driver"].y[train_mask])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

        if verbose and (epoch % max(1, epochs // 5) == 0 or epoch == epochs):
            model.eval()
            with torch.no_grad():
                test_logits, _ = model(data.x_dict, data.edge_index_dict)
                test_pred = test_logits[test_mask].argmax(dim=1)
                test_acc = (test_pred == data["driver"].y[test_mask]).float().mean().item()
            print(f"  Epoch {epoch:3d}/{epochs} | Loss {loss.item():.4f} | Test acc {test_acc:.4f}")

    return model, train_losses


def evaluate_graphsage(model, data, mask, label="GraphSAGE"):
    device = next(model.parameters()).device
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        logits, _ = model(data.x_dict, data.edge_index_dict)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    y_true = data["driver"].y.cpu().numpy()[mask.cpu().numpy()]
    y_prob = probs[mask.cpu().numpy(), 1]
    y_pred = y_prob >= 0.5
    acc = (y_pred == y_true).mean()
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n─── {label} Results (Test Set) ───")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Anomaly"], zero_division=0))
    print(f"AUC: {auc:.4f}  |  Accuracy: {acc:.4f}  |  F1: {f1:.4f}")
    return acc, auc, f1, y_true, y_prob, y_pred


# ── Train the main G-Full GraphSAGE model ────────────────────
in_channels = {ntype: hetero_graph[ntype].x.shape[1] for ntype in hetero_graph.node_types}
graphsage_model = HeterogeneousGraphSAGE(metadata=hetero_graph.metadata(), in_channels=in_channels)

graphsage_model, gnn_losses = train_graphsage(graphsage_model, hetero_graph, train_mask, test_mask)

plt.figure(figsize=(8, 4))
plt.plot(gnn_losses, label="Train loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("GraphSAGE Training Curve"); plt.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "graphsage_training_curve.png", dpi=150); plt.show()

acc_gnn, auc_gnn, f1_gnn, y_true_gnn, probs_gnn, preds_gnn = evaluate_graphsage(graphsage_model, hetero_graph, test_mask)

torch.save(graphsage_model.state_dict(), GRAPHSAGE_MODEL)
print(f"\n✅ GraphSAGE saved → {GRAPHSAGE_MODEL}")


In [ ]:
# ============================================================
# SECTION 13 — Phase VI: Full Model Evaluation & McNemar's Test (Obj 4)
# ============================================================

print_fallback_banner_if_needed()  # 🔧 FIX #3: loud tag if split fell back to random
# 🔧 FIX #5: the original script computed GraphSAGE's test set from
# `create_temporal_masks` (own leaky logic) and XGBoost's from
# `get_temporal_driver_split` (different leaky logic) — different sizes,
# different driver sets, and McNemar's test crashed on shape mismatch
# (253 vs 76). Every model now shares the same `driver_split` /
# `test_mask`, built once in Sections 7 & 10, over the SAME row order
# (`driver_features_norm`), so predictions are directly paired per driver.

def calculate_metrics(y_true, y_prob, k=PRECISION_AT_K):
    y_pred = y_prob >= 0.5
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    ap = average_precision_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan")
    f1 = f1_score(y_true, y_pred, zero_division=0)
    k_eff = min(k, len(y_prob))
    top_k_indices = np.argsort(y_prob)[::-1][:k_eff]
    prec_at_k = np.mean(y_true[top_k_indices]) if k_eff > 0 else float("nan")
    if len(set(y_true)) > 1:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        target_idx = np.where(tpr >= RECALL_TARGET)[0]
        fpr_at_target = fpr[target_idx[0]] if len(target_idx) > 0 else 1.0
    else:
        fpr_at_target = float("nan")
    return {"AUC": auc, "AP": ap, "F1": f1, f"Prec@{k}": prec_at_k, "FPR@90%Rec": fpr_at_target,
            "preds": y_pred, "probs": y_prob, "y_true": y_true}


def run_mcnemar_test(y_true, preds_a, preds_b, name_a, name_b):
    assert len(preds_a) == len(preds_b) == len(y_true), "McNemar's test requires paired, equal-length predictions"
    correct_a, correct_b = (preds_a == y_true), (preds_b == y_true)
    n_01 = np.sum(~correct_a & correct_b)
    n_10 = np.sum(correct_a & ~correct_b)
    if n_01 + n_10 == 0:
        statistic, p_value = 0.0, 1.0
    else:
        statistic = (abs(n_01 - n_10) - 1) ** 2 / (n_01 + n_10)
        p_value = chi2.sf(statistic, df=1)
    print(f"\n─── McNemar's Test: {name_a} vs {name_b} ───")
    print(f"  {name_a} correct / {name_b} wrong : {n_10}")
    print(f"  {name_b} correct / {name_a} wrong : {n_01}")
    print(f"  Chi2 Statistic : {statistic:.4f}  |  p-value: {p_value:.4e}")
    print("  Result         :", "Statistically significant (p < 0.05)" if p_value < 0.05
          else "No statistically significant difference (p ≥ 0.05)")


results = {}
results["LogReg"] = calculate_metrics(y_test_tab, probs_logreg)
results["XGBoost"] = calculate_metrics(y_test_tab, probs_xgb)
results["GraphSAGE"] = calculate_metrics(y_true_gnn, probs_gnn)  # same test_mask/order as tabular models
if len(y_test_seq) > 0:
    results["LSTM"] = calculate_metrics(y_test_seq, probs_lstm_seq)  # driver-disjoint but different row unit (sequences)

summary_df = pd.DataFrame([
    {"Model": name, "AUC-ROC": r["AUC"], "Avg Precision": r["AP"], "F1-Score": r["F1"],
     f"Precision@{PRECISION_AT_K}": r[f"Prec@{PRECISION_AT_K}"], "FPR@90% Recall": r["FPR@90%Rec"]}
    for name, r in results.items()
])
summary_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("═" * 70)
print("MODEL COMPARISON SUMMARY (Objective 4)")
print("═" * 70)
print(summary_df.to_string(index=False))

# GraphSAGE vs XGBoost are on the identical, aligned driver-level test set → valid McNemar's test
run_mcnemar_test(results["GraphSAGE"]["y_true"], results["GraphSAGE"]["preds"],
                  results["XGBoost"]["preds"], "GraphSAGE", "XGBoost")

plt.figure(figsize=(7, 6))
for name, r in results.items():
    if len(set(r["y_true"])) > 1:
        fpr, tpr, _ = roc_curve(r["y_true"], r["probs"])
        plt.plot(fpr, tpr, label=f"{name} (AUC={r['AUC']:.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Combined ROC Curves — Model Comparison (Objective 4)")
plt.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR / "roc_all_models.png", dpi=150)
plt.show()

print(f"\n✅ Evaluation complete — results saved to {RESULTS_DIR}")


In [ ]:
# ============================================================
# SECTION 14 — Phase VI: Graph Structure Ablation Study (Objective 7)
# ============================================================
# Uses the FIX #6 graph builder (Section 9), so G-Device no longer crashes,
# and the unified `train_mask`/`test_mask` (Section 10) for every variant,
# so results are comparable across variants and to Section 13's models.
# Fewer epochs are used here (ablation runs 4x models) — increase
# `ABLATION_EPOCHS` for a more thorough comparison once the pipeline is
# verified end-to-end.

ABLATION_EPOCHS = max(20, GNN_EPOCHS // 3)
variants = ["G-Spatial", "G-Temporal", "G-Device", "G-Full"]
ablation_results = []

for var in variants:
    print(f"\n{'='*60}\n Evaluating Graph Variant: {var}\n{'='*60}")
    # 🔧 FIX (spatial leakage): same window-restricted rides as the main graph
    graph_v, _ = build_heterogeneous_graph(rides_feat_windowed, driver_features_norm, variant=var)
    in_ch_v = {ntype: graph_v[ntype].x.shape[1] for ntype in graph_v.node_types}
    model_v = HeterogeneousGraphSAGE(metadata=graph_v.metadata(), in_channels=in_ch_v)

    model_v, _ = train_graphsage(model_v, graph_v, train_mask, test_mask, epochs=ABLATION_EPOCHS, verbose=False)
    acc_v, auc_v, f1_v, *_ = evaluate_graphsage(model_v, graph_v, test_mask, label=var)

    ablation_results.append({"Graph Variant": var, "Accuracy": acc_v, "AUC-ROC": auc_v, "F1-Score": f1_v})

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(RESULTS_DIR / "graph_ablation_comparison.csv", index=False)

print("\n" + "═" * 70)
print("GRAPH ABLATION STUDY RESULTS (Objective 7)")
print("═" * 70)
print(ablation_df.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.bar(ablation_df["Graph Variant"], ablation_df["AUC-ROC"], color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
plt.ylabel("AUC-ROC"); plt.title("Objective 7 — Which Edge Type Contributes Most?")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "graph_ablation_auc.png", dpi=150); plt.show()

print(f"\n✅ Ablation study complete — results saved to {RESULTS_DIR}")


In [ ]:
# ============================================================
# SECTION 15 — Phase VI: Graduated Anomaly Severity Score (Objective 8)
# ============================================================
# 🔧 FIX #8 (weights): Methodology §7.3 specifies GNN Prob 0.40 / Spatial
# Impossibility 0.30 / Temporal Persistence 0.20 / Device Switch 0.10.
# The original code used 0.40/0.25/0.20/0.15 and never actually computed
# "Temporal Persistence" (consecutive anomalous weeks) — it substituted a
# generic "behavioural deviation" (temporal_shift) term instead. Both are
# fixed below; weights now sum to 1.0 and match the spec exactly.
#
# 🔧 FIX #9 (units): scores are now computed from `driver_features_raw`
# (unscaled) so that `device_switch_rate`, `spatial_impossible_count`, etc.
# are real, interpretable values — not Z-scores that can be negative.

def compute_gass_scores(driver_features_raw: pd.DataFrame, gnn_probs: np.ndarray) -> pd.DataFrame:
    df = driver_features_raw.copy()
    p_gnn = gnn_probs

    # Spatial Impossibility (log-normalised per Methodology §7.3 scale column)
    sp_imp = df.get("spatial_impossible_count", pd.Series(0, index=df.index)).fillna(0)
    spatial_norm = np.log1p(sp_imp) / np.log1p(max(sp_imp.max(), 1))

    # Temporal Persistence — consecutive anomalous weeks, normalised to [0,1]
    persistence = df.get("temporal_persistence_weeks", pd.Series(0, index=df.index)).fillna(0)
    persistence_norm = persistence / max(persistence.max(), 1)

    # Device Switch Frequency, normalised to [0,1]
    dev_sw = df.get("device_switch_rate", pd.Series(0, index=df.index)).fillna(0)
    dev_norm = (dev_sw - dev_sw.min()) / (dev_sw.max() - dev_sw.min() + 1e-6)

    gass = (
        GASS_WEIGHT_GNN_PROB * p_gnn
        + GASS_WEIGHT_SPATIAL_IMPOSSIBLE * spatial_norm.values
        + GASS_WEIGHT_TEMPORAL_PERSISTENCE * persistence_norm.values
        + GASS_WEIGHT_DEVICE_SWITCH * dev_norm.values
    )
    df["GASS"] = gass

    conditions = [df["GASS"] < GASS_THRESHOLD_LOW,
                  (df["GASS"] >= GASS_THRESHOLD_LOW) & (df["GASS"] < GASS_THRESHOLD_HIGH),
                  df["GASS"] >= GASS_THRESHOLD_HIGH]
    df["Severity_Tier"] = np.select(conditions, ["Low", "Medium", "High"], default="Low")
    return df


# Score every driver (not just the test split) — GASS is an operational
# tool applied to the whole fleet, matching Methodology §7.3.
graphsage_model.eval()
with torch.no_grad():
    all_logits, _ = graphsage_model(hetero_graph.x_dict, hetero_graph.edge_index_dict)
    all_gnn_probs = torch.softmax(all_logits, dim=1).cpu().numpy()[:, 1]

# driver_features_raw rows are in the same order as the graph's driver nodes
# (both built from driver_features_norm / driver_features_raw's driver_id order)
scored_df = compute_gass_scores(driver_features_raw, all_gnn_probs)
scored_df.to_csv(RESULTS_DIR / "driver_gass_scores.csv", index=False)

tier_counts = scored_df["Severity_Tier"].value_counts()
print("─── GASS Severity Tier Breakdown (Objective 8) ───")
print(tier_counts.to_string())
print(f"\nWeights used — GNN Prob: {GASS_WEIGHT_GNN_PROB}, Spatial: {GASS_WEIGHT_SPATIAL_IMPOSSIBLE}, "
      f"Temporal Persistence: {GASS_WEIGHT_TEMPORAL_PERSISTENCE}, Device Switch: {GASS_WEIGHT_DEVICE_SWITCH}")

plt.figure(figsize=(7, 5))
tier_counts.reindex(["Low", "Medium", "High"]).fillna(0).plot(kind="bar", color=["green", "orange", "red"])
plt.title("Graduated Anomaly Severity Score (GASS) Tiers")
plt.xlabel("Severity Tier"); plt.ylabel("Number of Drivers")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "gass_tier_distribution.png", dpi=150); plt.show()

print(f"\n✅ Severity scoring complete — results saved to {RESULTS_DIR}")


In [ ]:
# ============================================================
# SECTION 16 — Phase VI: Explainability (Objective 6)
# ============================================================
# 🔧 FIX #9 (again): cards now read from `driver_features_raw` (real units).
# 🔧 FIX #10: the original card template hardcoded "(High Risk)" and
# "ANOMALOUS DETECTED" for every driver in the top-5 by probability, even
# when their actual GASS tier was Low/Medium. The card now reports the
# REAL GASS tier and score, computed in Section 15, so the label always
# matches the system's own severity system (important for the legal
# defensibility goal — Methodology §7.2 / xFraud [7]).

def generate_explanation_card(driver_id: str, gass_row: pd.Series) -> str:
    risk_tier = gass_row.get("Severity_Tier", "Unknown")
    gass_score = gass_row.get("GASS", float("nan"))
    flagged = "ANOMALOUS — FLAGGED" if risk_tier in ("Medium", "High") else "Within normal range"

    return f"""
{'='*80}
DRIVER IDENTITY MISUSE EXPLANATION CARD (GNNExplainer-style)
{'='*80}
Driver Account ID     : {driver_id}
GASS Score            : {gass_score:.4f}   |   Risk Tier: {risk_tier}
Flagged Status         : {flagged}

SUMMARY OF BEHAVIOURAL INCONSISTENCIES:
{'-'*80}
1. Spatio-Temporal Inconsistency:
   - Temporal Centroid Shift : {gass_row.get('temporal_shift', 0.0):.2f} hours from baseline profile.
   - Spatial Home Cluster    : Baseline pickup borough was {gass_row.get('baseline_borough', 'Unknown')}.
   - Spatial Impossibility   : {int(gass_row.get('spatial_impossible_count', 0))} impossible-travel event(s) detected.
   - Anomalous Weeks (streak): {int(gass_row.get('temporal_persistence_weeks', 0))} consecutive week(s).

2. Device Fingerprint Anomaly:
   - Active Device Count     : {int(gass_row.get('num_devices', 1))} unique device(s).
   - Device Switch Rate      : {gass_row.get('device_switch_rate', 0.0):.2f} (fraction of rides on a non-primary device).

3. Graph Relational Evidence (GraphSAGE Subgraph Contribution):
   - Driver ↔ Driver Spatial/Temporal Overlap : connected to other driver accounts operating
     in the same zones and/or overlapping shift hours.
   - Driver ↔ Device Co-usage                 : shared device signature contributes to the flag.

RECOMMENDED INVESTIGATOR ACTION:
{'-'*80}
{"Require real-time biometric verification / selfie prompt upon next login. Inspect login logs for IP/MAC duplication across accounts."
 if risk_tier == "High" else
 "Monitor account; re-evaluate at next scoring cycle." if risk_tier == "Medium" else
 "No action required at this time."}
{'='*80}
"""


# Top-5 by GASS score (not raw GNN probability) — consistent with the
# system's own severity ranking, fixing the "top-5 always High Risk" bug.
top5 = scored_df.sort_values("GASS", ascending=False).head(5)

explanation_cards = []
for _, row in top5.iterrows():
    card = generate_explanation_card(row["driver_id"], row)
    explanation_cards.append(card)
    print(card)

with open(RESULTS_DIR / "explanation_cards.txt", "w") as f:
    f.write("\n\n".join(explanation_cards))

print(f"✅ Explanation cards saved to {RESULTS_DIR / 'explanation_cards.txt'}")


In [ ]:
# ============================================================
# SECTION 17 — Exploratory Data Analysis & Visualisations
# ============================================================

print(f"Dataset shape : {rides_feat.shape}")
print(f"Date range    : {rides_feat['tpep_pickup_datetime'].min()} → {rides_feat['tpep_pickup_datetime'].max()}")
print(f"Anomaly rate  : {100 * rides_feat['is_anomaly'].mean():.2f} %\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
anomaly_counts = rides_feat["is_anomaly"].value_counts()
axes[0].bar(["Normal", "Anomaly"], [anomaly_counts.get(0, 0), anomaly_counts.get(1, 0)],
            color=["steelblue", "tomato"])
axes[0].set_title("Ride Count by Label"); axes[0].set_ylabel("Count")

type_counts = rides_feat["anomaly_type"].value_counts()
axes[1].pie(type_counts.values, labels=type_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("muted", len(type_counts)))
axes[1].set_title("Anomaly Types Breakdown")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "01_anomaly_distribution.png", dpi=150); plt.show()

plt.figure(figsize=(10, 5))
for atype in rides_feat["anomaly_type"].unique():
    subset = rides_feat[rides_feat["anomaly_type"] == atype]["trip_distance"].clip(0, 25)
    plt.hist(subset, bins=50, alpha=0.5, label=atype)
plt.xlabel("Trip Distance (miles)"); plt.ylabel("Frequency")
plt.title("Trip Distance Distribution by Anomaly Type"); plt.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "02_distance_distribution.png", dpi=150); plt.show()

print(f"✅ EDA visualisations saved to {RESULTS_DIR}")


In [ ]:
# ============================================================
# SECTION 18 — Final Summary
# ============================================================

print_fallback_banner_if_needed()  # 🔧 FIX #3: loud tag if split fell back to random

print("═" * 70)
print(" DIMD FRAMEWORK — RUN SUMMARY")
print("═" * 70)
print(f"\nData source                            : {'REAL NYC TLC 2023 CSV' if USING_REAL_DATA else 'SYNTHETIC DEMO DATA (not the real 43M-row dataset — see Section 2 warning)'}")
print(f"Label-leak guard (FIX #12)             : {[c for c in LABEL_DERIVED_COLS if c in driver_features_norm.columns]} excluded from baseline FEATURE_COLS")
print(f"\nObjective 1 (Detection Framework)     : pipeline ran end-to-end, driver split leak-free")
print(f"Objective 2 (Feature Engineering)      : {driver_features_raw.shape[1]} driver-level features")
print(f"Objective 3 (Graph Construction)       : {len(hetero_graph.node_types)} node types, {len(hetero_graph.edge_types)} edge types")
print(f"Objective 4 (GNN vs Traditional)       : see model_comparison.csv")
print(summary_df.to_string(index=False))
print(f"\nObjective 5 (Synthetic Anomaly Inj.)   : {100*rides_feat['is_anomaly'].mean():.1f}% ride-level anomaly rate")
print(f"Objective 6 (Explainability)           : explanation_cards.txt ({len(explanation_cards)} cards)")
print(f"Objective 7 (Graph Structure Compare.) : see graph_ablation_comparison.csv")
print(ablation_df.to_string(index=False))
print(f"\nObjective 8 (Severity Scoring)         : GASS tiers —")
print(tier_counts.to_string())

print(f"\nAll outputs saved under: {RESULTS_DIR}")
print("=" * 70)


In [ ]:
import os
from google.colab import files

output_filename = "dimd_project.zip"
project_dir = "/content/dimd_project"

# Create the zip archive
!zip -r "{output_filename}" "{project_dir}"

# Download the zip file
files.download(output_filename)
